## Step 0 — Fix dependencies and restart runtime
Run this cell once. It will automatically restart the kernel.

In [ ]:
# Upgrade numpy + scipy together so their ABIs match on Python 3.12
# Also install everything else needed before the restart.
import subprocess, sys

pkgs = [
    "numpy>=2.0",
    "scipy>=1.14.0",
    "scikit-image>=0.22",
    "SimpleITK>=2.3",
    "h5py>=3.10",
    "pydicom>=2.4",
    "opencv-python-headless>=4.9",
    "pandas>=2.0",
    "matplotlib>=3.8",
    "seaborn",
    "tqdm",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U"] + pkgs,
    check=True
)
print("Packages installed. Restarting runtime now...")
import os, signal
os.kill(os.getpid(), signal.SIGKILL)  # hard restart


## Step 0b — Sanity check (run after restart)

In [1]:
import numpy as np
import scipy, cv2, SimpleITK as sitk, h5py, pydicom
print(f"numpy  {np.__version__}")
print(f"scipy  {scipy.__version__}")
print(f"cv2    {cv2.__version__}")
print(f"sitk   {sitk.Version()}")
print(f"h5py   {h5py.__version__}")
print(f"pydicom {pydicom.__version__}")
print("All imports OK.")


numpy  2.4.6
scipy  1.17.1
cv2    4.13.0
sitk   SimpleITK Version: 2.5.5 (ITK 5.4)
Compiled: May 12 2026 17:19:38

h5py   3.16.0
pydicom 3.0.2
All imports OK.


## Step 1 — Mount Google Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
import os
print(os.listdir("/content/drive/MyDrive"))


['PROJECTS', 'Sem 7', 'Sem 8', 'Dr. Karali Patra - IITP - Winter Internship', 'extra', 'Colab Notebooks', 'BOOKS', 'WhatsApp Image 2026-03-25 at 19.35.50.jpeg', 'Untitled document.gdoc', 'CrossModal_DR_Checkpoints', 'dataset ']


In [4]:
print(os.path.exists("/content/drive/MyDrive/dataset "))


True


In [5]:
print(os.listdir("/content/drive/MyDrive/dataset "))


['8026660', 'NCI-ISBI-2013-Prostate-Challenge-Training', 'metadata', 'prostate_3t', 'prostate_diagnosis']


In [6]:
for item in os.listdir("/content/drive/MyDrive/dataset "):
    full_path = os.path.join("/content/drive/MyDrive/dataset ", item)
    print(f"{'[DIR]' if os.path.isdir(full_path) else '[FILE]'} {item}")


[DIR] 8026660
[DIR] NCI-ISBI-2013-Prostate-Challenge-Training
[DIR] metadata
[DIR] prostate_3t
[DIR] prostate_diagnosis


In [7]:
dataset_path = "/content/drive/MyDrive/dataset "
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)
    for file in files[:5]:
        print(f"{subindent}{file}")
    if len(files) > 5:
        print(f"{subindent}... and {len(files)-5} more files")


dataset /
    8026660/
        LICENSE.TXT
        training_data/
            Case01.raw
            Case02.mhd
            Case00_segmentation.raw
            Case01_segmentation.raw
            Case01.mhd
            ... and 195 more files
        test_data/
            Case00.raw
            Case02_segmentation.mhd
            Case03.raw
            Case04.mhd
            Case01.raw
            ... and 115 more files
        livechallenge_test_data/
            Case00_segmentation.raw
            Case01_segmentation.mhd
            Case01_segmentation.raw
            Case02.mhd
            Case01.raw
            ... and 75 more files
    NCI-ISBI-2013-Prostate-Challenge-Training/
        Training/
            ProstateDx-01-0023.nrrd
            Prostate3T-01-0006.nrrd
            Prostate3T-01-0005.nrrd
            ProstateDx-01-0052.nrrd
            ProstateDx-01-0059.nrrd
            ... and 55 more files
    metadata/
        metadata.csv
    prostate_3t/
        Prostate3T-01-00

In [8]:
base = "/content/drive/MyDrive/dataset /prostate_3t"
for root, dirs, files in os.walk(base):
    if files:
        size = sum(os.path.getsize(os.path.join(root, f)) for f in files)
        print(f"{os.path.relpath(root, base)}: {len(files)} files, {size/1024/1024:.1f} MB")
        for f in files[:3]:
            print(f"    -> {f}")
        if len(files) > 3:
            print(f"    ... and {len(files)-3} more")


Prostate3T-01-0025/69630/29351: 19 files, 3.8 MB
    -> c5837a9b-ba44-486c-9984-44d03ce37cec.dcm
    -> 0b4812f7-4760-4e8b-8f86-806a3fa7cb38.dcm
    -> b04a8d03-f32c-4088-8e19-bc1593ce63bb.dcm
    ... and 16 more
Prostate3T-01-0030/32765/22882: 20 files, 4.0 MB
    -> 4a0f1f38-a448-44f5-9c36-caeccef638c8.dcm
    -> 489e3342-8ccd-4df1-9552-b24dcbafb842.dcm
    -> 1e3d3383-a46f-4534-be81-b6869d539cc9.dcm
    ... and 17 more
Prostate3T-01-0028/64995/83650: 15 files, 3.0 MB
    -> 22864ccf-4761-43ca-850d-6c7dc0f1c40f.dcm
    -> 1e2e3f3b-5f55-4aeb-b77d-33ce89784245.dcm
    -> 68e00231-718e-43ac-9a5a-329438bf8547.dcm
    ... and 12 more
Prostate3T-01-0027/60968/89418: 20 files, 4.0 MB
    -> 4814fb52-38ad-4a91-9646-0b04a014cf2e.dcm
    -> e38f0d3b-4002-4d80-8361-2a19ff1da8ca.dcm
    -> ba4fd12c-3e0c-47fe-aeb5-25e0472dd00d.dcm
    ... and 17 more
Prostate3T-01-0029/88536/30107: 20 files, 4.0 MB
    -> 02363031-75f1-4e8e-b70d-ba4da6707911.dcm
    -> e7523b79-6a4d-4d8c-9b47-73a5469120a1.dcm
   

In [9]:
base = "/content/drive/MyDrive/dataset /prostate_diagnosis"
for root, dirs, files in os.walk(base):
    if files:
        size = sum(os.path.getsize(os.path.join(root, f)) for f in files)
        print(f"{os.path.relpath(root, base)}: {len(files)} files, {size/1024/1024:.1f} MB")
        for f in files[:3]:
            print(f"    -> {f}")
        if len(files) > 3:
            print(f"    ... and {len(files)-3} more")


ProstateDx-01-0080/37331/47262: 32 files, 9.9 MB
    -> f400df15-cddc-4c7a-af6a-e199ff08ebbd.dcm
    -> 4c041516-3567-4fec-bb4c-a9329fb501f4.dcm
    -> 18b91ff0-5867-4ea7-a6bb-1447feaa47f0.dcm
    ... and 29 more
ProstateDx-01-0082/62402/04339: 32 files, 11.5 MB
    -> 1a25cb06-311c-495b-bfe6-00d07a2f3ecd.dcm
    -> 06bf80d8-79e0-4f6a-95bd-26731769b0fb.dcm
    -> f3763b70-3bc6-4533-a809-7fd15b4cce72.dcm
    ... and 29 more
ProstateDx-01-0075/22755/56170: 34 files, 10.5 MB
    -> f845e8f7-f198-496f-a430-af24264716e0.dcm
    -> e78f67d9-0674-40c1-9dbc-b65adecc734d.dcm
    -> 487ad674-b24a-4aaf-9a38-918a48547917.dcm
    ... and 31 more
ProstateDx-01-0076/47756/41667: 32 files, 9.9 MB
    -> 615e05bb-8eac-49c0-9407-581fb8466800.dcm
    -> fcb83776-c4c7-495b-8a33-dfd173c50294.dcm
    -> e3f6cdb7-27f4-468f-9ffc-063b0e2f277d.dcm
    ... and 29 more
ProstateDx-01-0063/03810/33058: 32 files, 9.9 MB
    -> 15593f83-e3d9-4f46-9330-28c9e99e4f6b.dcm
    -> 137a5edb-2514-41bf-9df9-2d5e28d0426e.dcm
 

In [11]:
dataset_path = "/content/drive/MyDrive/dataset "
total_size = total_files = 0
for root, dirs, files in os.walk(dataset_path):
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    if files:
        size = sum(os.path.getsize(os.path.join(root, f)) for f in files)
        total_size += size
        total_files += len(files)
        rel = os.path.relpath(root, dataset_path)
        print(f"{rel}: {len(files)} files, {size/1024/1024:.1f} MB")
print(f"GRAND TOTAL: {total_files} files, {total_size/1024/1024/1024:.2f} GB")


8026660: 1 files, 0.0 MB
8026660/training_data: 200 files, 750.1 MB
8026660/test_data: 120 files, 445.0 MB
8026660/livechallenge_test_data: 80 files, 283.2 MB
NCI-ISBI-2013-Prostate-Challenge-Training/Training: 60 files, 0.5 MB
metadata: 1 files, 0.0 MB
prostate_3t/Prostate3T-01-0025/69630/29351: 19 files, 3.8 MB
prostate_3t/Prostate3T-01-0030/32765/22882: 20 files, 4.0 MB
prostate_3t/Prostate3T-01-0028/64995/83650: 15 files, 3.0 MB
prostate_3t/Prostate3T-01-0027/60968/89418: 20 files, 4.0 MB
prostate_3t/Prostate3T-01-0029/88536/30107: 20 files, 4.0 MB
prostate_3t/Prostate3T-01-0026/41751/03388: 20 files, 4.0 MB
prostate_3t/Prostate3T-01-0024/84925/37007: 20 files, 4.0 MB
prostate_3t/Prostate3T-01-0018/37976/35163: 20 files, 4.0 MB
prostate_3t/Prostate3T-01-0022/78894/51333: 20 files, 4.0 MB
prostate_3t/Prostate3T-01-0019/88202/28931: 18 files, 2.3 MB
prostate_3t/Prostate3T-01-0021/27772/75076: 20 files, 4.0 MB
prostate_3t/Prostate3T-01-0020/58257/50594: 20 files, 4.0 MB
prostate_3t/Pr

In [13]:
import SimpleITK as sitk
import numpy as np

nrrd_dir = "/content/drive/MyDrive/dataset /NCI-ISBI-2013-Prostate-Challenge-Training/Training"
files = sorted(os.listdir(nrrd_dir))
print(f"Total .nrrd files: {len(files)}")
print("First 10 files:")
for f in files[:10]:
    print(f" {f}")

sample = sitk.ReadImage(os.path.join(nrrd_dir, files[0]))
print(f"Sample: {files[0]}")
print(f"  Size: {sample.GetSize()}")
print(f"  Spacing: {sample.GetSpacing()}")
print(f"  Pixel type: {sample.GetPixelIDTypeAsString()}")
arr = sitk.GetArrayFromImage(sample)
print(f"  Array shape: {arr.shape}")
print(f"  Unique values: {np.unique(arr)}")


Total .nrrd files: 60
First 10 files:
 Prostate3T-01-0001.nrrd
 Prostate3T-01-0002.nrrd
 Prostate3T-01-0003.nrrd
 Prostate3T-01-0004.nrrd
 Prostate3T-01-0005.nrrd
 Prostate3T-01-0006.nrrd
 Prostate3T-01-0007.nrrd
 Prostate3T-01-0008.nrrd
 Prostate3T-01-0009.nrrd
 Prostate3T-01-0010.nrrd
Sample: Prostate3T-01-0001.nrrd
  Size: (320, 320, 15)
  Spacing: (0.6000000238418621, 0.6000000238418624, 4.000002498448475)
  Pixel type: 16-bit signed integer
  Array shape: (15, 320, 320)
  Unique values: [0 1 2]


In [14]:
import pydicom

dcm_base = "/content/drive/MyDrive/dataset /prostate_3t/Prostate3T-01-0001/79232/49698"
dcm_files = sorted(os.listdir(dcm_base))
dcm = pydicom.dcmread(os.path.join(dcm_base, dcm_files[0]))
print(f"Patient ID: {dcm.PatientID}")
print(f"Modality: {dcm.Modality}")
print(f"Rows x Cols: {dcm.Rows} x {dcm.Columns}")
print(f"Manufacturer: {getattr(dcm, 'Manufacturer', 'N/A')}")
print(f"MagneticFieldStrength: {getattr(dcm, 'MagneticFieldStrength', 'N/A')}")
print(f"SliceThickness: {getattr(dcm, 'SliceThickness', 'N/A')}")
print(f"Number of slices in folder: {len(dcm_files)}")


Patient ID: Prostate3T-01-0001
Modality: MR
Rows x Cols: 320 x 320
Manufacturer: SIEMENS
MagneticFieldStrength: 3
SliceThickness: 4
Number of slices in folder: 15


In [16]:
mhd_path = "/content/drive/MyDrive/dataset /8026660/training_data/Case00.mhd"
img = sitk.ReadImage(mhd_path)
print(f"Size: {img.GetSize()}")
print(f"Spacing: {img.GetSpacing()}")
print(f"Pixel type: {img.GetPixelIDTypeAsString()}")
arr = sitk.GetArrayFromImage(img)
print(f"Array shape: {arr.shape}")
print(f"Value range: {arr.min()} to {arr.max()}")

seg_path = "/content/drive/MyDrive/dataset /8026660/training_data/Case00_segmentation.mhd"
if os.path.exists(seg_path):
    seg = sitk.ReadImage(seg_path)
    seg_arr = sitk.GetArrayFromImage(seg)
    print(f"Segmentation found!")
    print(f"Seg shape: {seg_arr.shape}")
    print(f"Seg unique values: {np.unique(seg_arr)}")


Size: (512, 512, 47)
Spacing: (0.273438, 0.273437, 2.20001)
Pixel type: 16-bit signed integer
Array shape: (47, 512, 512)
Value range: 0 to 12435
Segmentation found!
Seg shape: (47, 512, 512)
Seg unique values: [0 1]


In [18]:
nrrd_dir = "/content/drive/MyDrive/dataset /NCI-ISBI-2013-Prostate-Challenge-Training/Training"
nrrd_files = sorted(os.listdir(nrrd_dir))
prostate3t_cases  = sorted(os.listdir("/content/drive/MyDrive/dataset /prostate_3t"))
prostateDx_cases  = sorted(os.listdir("/content/drive/MyDrive/dataset /prostate_diagnosis"))

nrrd_3t = [f for f in nrrd_files if "Prostate3T" in f]
nrrd_dx = [f for f in nrrd_files if "ProstateDx" in f]
print(f"NRRD masks for Prostate3T : {len(nrrd_3t)}")
print(f"NRRD masks for ProstateDx : {len(nrrd_dx)}")
print(f"DICOM cases in prostate_3t          : {len(prostate3t_cases)}")
print(f"DICOM cases in prostate_diagnosis   : {len(prostateDx_cases)}")

nrrd_3t_ids  = {f.replace(".nrrd", "") for f in nrrd_3t}
nrrd_dx_ids  = {f.replace(".nrrd", "") for f in nrrd_dx}
dicom_3t_ids = set(prostate3t_cases)
dicom_dx_ids = set(prostateDx_cases)

matched_3t = nrrd_3t_ids & dicom_3t_ids
matched_dx = nrrd_dx_ids & dicom_dx_ids
print(f"Matched 3T  (image+mask): {len(matched_3t)}")
print(f"Matched Dx  (image+mask): {len(matched_dx)}")
print(f"3T  unmatched DICOM (no mask): {dicom_3t_ids - nrrd_3t_ids}")
print(f"Dx  unmatched DICOM (no mask): {dicom_dx_ids - nrrd_dx_ids}")


NRRD masks for Prostate3T : 30
NRRD masks for ProstateDx : 30
DICOM cases in prostate_3t          : 30
DICOM cases in prostate_diagnosis   : 30
Matched 3T  (image+mask): 30
Matched Dx  (image+mask): 29
3T  unmatched DICOM (no mask): set()
Dx  unmatched DICOM (no mask): {'ProstateDx-01-0006'}


## Step 2 — Clone FedLPPA repository

In [19]:
!git clone https://github.com/llmir/FedLPPA.git
%cd FedLPPA


Cloning into 'FedLPPA'...
remote: Enumerating objects: 653, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 653 (delta 24), reused 0 (delta 0), pack-reused 609 (from 1)
Receiving objects: 100% (653/653), 6.62 MiB | 29.85 MiB/s, done.
Resolving deltas: 100% (309/309), done.
/content/FedLPPA


In [21]:
import os, collections

print("=" * 70)
print("COMPLETE FILE TREE OF FedLPPA REPO")
print("=" * 70)

all_files = []
for root, dirs, files in os.walk("."):
    dirs[:] = [d for d in dirs if not d.startswith(".") and d != "__pycache__"]
    level = root.replace(".", "").count(os.sep)
    indent = "  " * level
    if root != ".":
        print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        print(f"{subindent}{f}  ({size/1024:.1f} KB)")
        all_files.append(fpath)

print(f"Total files: {len(all_files)}")


COMPLETE FILE TREE OF FedLPPA REPO
  tree_filter-0.1-cp39-cp39-linux_x86_64.whl  (95.8 KB)
  README.md  (10.4 KB)
  .gitignore  (3.0 KB)
  fedlppa.yaml  (5.7 KB)
  code_v4/
    test.sh  (2.4 KB)
    flower_common_v4.py  (91.1 KB)
    flower_pCE_2D.py  (35.7 KB)
    train.sh  (5.3 KB)
    flower_common.py  (87.7 KB)
    flower_pCE_2D_v4.py  (38.9 KB)
    flower_common_v4_addprostate.py  (91.2 KB)
    flower_command.sh  (2.0 KB)
    val_2D.py  (13.9 KB)
    test_client4onemod_FL_Personalize.py  (20.5 KB)
    flower_pCE_2D_v4_FedLPPA.py  (38.9 KB)
    networks/
      unet_3D.py  (3.6 KB)
      networks_other.py  (20.2 KB)
      net_factory.py  (4.9 KB)
      utils.py  (18.2 KB)
      attention_unet.py  (6.3 KB)
      net_factory_3d.py  (0.8 KB)
      unet.py  (85.3 KB)
      VoxResNet.py  (3.7 KB)
      vnet.py  (9.6 KB)
      discriminator.py  (3.2 KB)
      efficient_encoder.py  (17.7 KB)
      efficientunet.py  (8.0 KB)
      attention.py  (3.1 KB)
      grid_attention_layer.py  (16.6 

In [22]:
TEXT_EXTENSIONS = {".py", ".sh", ".yaml", ".yml", ".txt", ".md",
                   ".json", ".cfg", ".conf", ".ini", ".toml", ".rst"}

print("" + "=" * 70)
print("CONTENTS OF ALL TEXT FILES")
print("=" * 70)

for fpath in sorted(all_files):
    ext = os.path.splitext(fpath)[1].lower()
    if ext not in TEXT_EXTENSIONS:
        continue
    print(f"{'#' * 70}")
    print(f"FILE: {fpath}")
    print("#" * 70)
    try:
        with open(fpath, "r", encoding="utf-8", errors="replace") as f:
            print(f.read())
    except Exception as e:
        print(f"[Could not read: {e}]")


CONTENTS OF ALL TEXT FILES
######################################################################
FILE: ./README.md
######################################################################
# FedLPPA: Learning Personalized Prompt and Aggregation for Federated Weakly-supervised Medical Image Segmentation
The official implementation of the paper: **FedLPPA: Learning Personalized Prompt and Aggregation for Federated Weakly-supervised Medical Image Segmentation**([arxiv](https://arxiv.org/abs/2402.17502), [Early Access](https://ieeexplore.ieee.org/document/10721443))

# 🔔News
- 2024-10-15, 🎉🎉 Our paper "[FedLPPA: Learning Personalized Prompt and Aggregation for Federated Weakly-supervised Medical Image Segmentation](https://arxiv.org/pdf/2402.17502)" has been accepted by **IEEE Transactions on Medical Imaging (TMI)**.

# 💡Framework
![TEL](image/framework.png)

# 🌟 Motivation

**Particularly Valuable for Addressing the Problems**:  
   - Cross-institutional collaboration barriers ("data silos"

In [23]:
py_files = [f for f in all_files if f.endswith(".py")]
print(f"Total .py files: {len(py_files)}")

by_dir = collections.defaultdict(list)
for f in py_files:
    by_dir[os.path.dirname(f)].append(os.path.basename(f))

for d, files in sorted(by_dir.items()):
    print(f"{d}/")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(d, f))
        print(f"    {f}  ({size/1024:.1f} KB)")


Total .py files: 100
./code_v4/
    flower_common.py  (87.7 KB)
    flower_common_v4.py  (91.1 KB)
    flower_common_v4_addprostate.py  (91.2 KB)
    flower_pCE_2D.py  (35.7 KB)
    flower_pCE_2D_v4.py  (38.9 KB)
    flower_pCE_2D_v4_FedLPPA.py  (38.9 KB)
    test_client4onemod_FL_Personalize.py  (20.5 KB)
    val_2D.py  (13.9 KB)
./code_v4/dataloaders/
    Unet_pCE.py  (11.9 KB)
    acdc_pseudo_label_random_walker.py  (2.4 KB)
    dataset.py  (15.8 KB)
    dataset_CL.py  (15.3 KB)
    dataset_ft.py  (15.4 KB)
    dataset_fully.py  (7.8 KB)
    dataset_rw.py  (15.3 KB)
    dataset_s2l.py  (12.4 KB)
    dataset_semi.py  (8.1 KB)
    icctw_data_processing.py  (11.6 KB)
    odocfaz_data_processing.py  (11.6 KB)
    utils.py  (6.8 KB)
./code_v4/networks/
    VoxResNet.py  (3.7 KB)
    attention.py  (3.1 KB)
    attention_unet.py  (6.3 KB)
    discriminator.py  (3.2 KB)
    efficient_encoder.py  (17.7 KB)
    efficientunet.py  (8.0 KB)
    encoder_tool.py  (6.8 KB)
    grid_attention_layer.

## Step 3 — Install flwr (Flower federated learning framework)
Done separately from numpy/scipy to avoid conflicts.

In [25]:
# ============================================================
# CELL: Install flwr (fixed for Python 3.12 Colab)
# ============================================================
# grpcio 1.46 has no py3.12 wheel — use 1.59+ which does.
# flwr 1.0.0 caps at py<3.11 — use 1.4.0 which is the last
# version keeping the raw Client / FitRes / EvaluateRes API
# that FedLPPA's flower_pCE_2D_v4_FedLPPA.py depends on.
!pip install -q \
    "flwr==1.4.0" \
    "grpcio>=1.59.0,<2.0" \
    "protobuf>=3.20.0,<4.0" \
    --quiet

import flwr as fl
print(f"flwr version: {fl.__version__}")

# Quick smoke-test that the exact names FedLPPA imports exist
from flwr.common import (
    GetParametersRes, Status, FitRes, EvaluateRes,
    ndarrays_to_parameters, parameters_to_ndarrays,
    GetPropertiesIns, GetPropertiesRes,
)
from flwr.server.strategy import FedAvg
print("All required flwr symbols present — OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.2/157.2 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-cloud-language 2.20.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-cloud-speech 2.39.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-datastore 2.24.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires 

In [26]:
import os
os.chdir("/content/FedLPPA")
!pip install -q tree_filter-0.1-cp39-cp39-linux_x86_64.whl 2>/dev/null || \n echo "Wheel not compatible with current Python — tree filter will be skipped at runtime."


/bin/bash: line 1: n: command not found


## Step 4 — Preprocess NCI-ISBI 2013 (Domains 1 & 2)

| Domain | Source | Scanner | Annotation |
|--------|--------|---------|------------|
| Domain1 | prostate_3t | Siemens 3T | block |
| Domain2 | prostate_diagnosis | Philips 1.5T | keypoint |

In [5]:
import os, glob, h5py, cv2
import numpy as np
import SimpleITK as sitk
import pydicom
import torch
import torch.nn.functional as F
from concurrent.futures import ThreadPoolExecutor, as_completed

TARGET_SIZE = 384
TRAIN_RATIO = 0.80
H5_BASE     = "/content/data/Prostate_h5"
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ── GPU helpers ──────────────────────────────────────────────────

def normalize_vol_gpu(vol_np):
    """Percentile normalisation on GPU, returns float32 numpy."""
    t = torch.from_numpy(vol_np).to(DEVICE)
    lo = torch.quantile(t, 0.01)
    hi = torch.quantile(t, 0.99)
    t  = torch.clamp(t, lo, hi)
    t  = (t - t.min()) / (t.max() - t.min() + 1e-8)
    return t.cpu().numpy()

def rs_img_gpu(slc, n=TARGET_SIZE):
    """Bilinear resize on GPU → float32 numpy (H,W)."""
    t = torch.from_numpy(slc.astype(np.float32)).to(DEVICE)
    t = t.unsqueeze(0).unsqueeze(0)                         # (1,1,H,W)
    t = F.interpolate(t, size=(n, n), mode="bilinear",
                      align_corners=False)
    return t.squeeze().cpu().numpy()

def rs_msk_gpu(slc, n=TARGET_SIZE):
    """Nearest-neighbor resize on GPU → uint8 numpy (H,W)."""
    t = torch.from_numpy(slc.astype(np.float32)).to(DEVICE)
    t = t.unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=(n, n), mode="nearest")
    return t.squeeze().cpu().numpy().astype(np.uint8)

# ── DICOM / NRRD loaders (CPU, I/O bound) ───────────────────────

def load_dicom_volume(case_dir):
    best_folder, best_count = None, 0
    for root, _, files in os.walk(case_dir):
        dcms = [f for f in files if not f.startswith(".")]
        if len(dcms) > best_count:
            try:
                pydicom.dcmread(os.path.join(root, dcms[0]),
                                stop_before_pixels=True)
                best_folder, best_count = root, len(dcms)
            except Exception:
                pass
    if best_folder is None:
        return None
    slices = []
    for fp in sorted(os.listdir(best_folder)):
        try:
            slices.append(pydicom.dcmread(os.path.join(best_folder, fp)))
        except Exception:
            pass
    slices.sort(key=lambda x: float(
        getattr(x, "ImagePositionPatient", [0, 0, 0])[2]))
    return np.stack([s.pixel_array.astype(np.float32) for s in slices])

def load_nrrd_mask(path):
    return (sitk.GetArrayFromImage(sitk.ReadImage(path)) > 0).astype(np.uint8)

# ── Per-case worker (called from thread pool) ────────────────────

def process_case(args):
    """Returns list of (fname, img_arr, msk_arr) tuples for one case."""
    cid, cdir, nrrd_path = args
    vol = load_dicom_volume(cdir)
    if vol is None:
        return cid, []
    msk = load_nrrd_mask(nrrd_path)
    n   = min(vol.shape[0], msk.shape[0])
    vol = normalize_vol_gpu(vol[:n])
    msk = msk[:n]
    results = []
    for i in range(n):
        if msk[i].sum() == 0:
            continue
        results.append((
            f"{cid}_s{i:03d}.h5",
            rs_img_gpu(vol[i]),
            rs_msk_gpu(msk[i]),
        ))
    return cid, results

# ── Main site processor ──────────────────────────────────────────

def process_nci_site(dicom_base, nrrd_dir, domain, out_base,
                     ratio=TRAIN_RATIO, workers=4):
    cases  = sorted(os.listdir(dicom_base))
    paired = []
    for cid in cases:
        nrrd = os.path.join(nrrd_dir, f"{cid}.nrrd")
        if os.path.exists(nrrd):
            paired.append((cid, os.path.join(dicom_base, cid), nrrd))

    print(f"{domain}: {len(paired)} matched cases")
    n_tr   = max(1, int(len(paired) * ratio))
    splits = {"train": paired[:n_tr], "test": paired[n_tr:]}

    for split, lst in splits.items():
        out_dir = os.path.join(out_base, domain, split)
        os.makedirs(out_dir, exist_ok=True)
        cnt = 0
        # Parallel I/O, serial GPU (CUDA ops are already async)
        with ThreadPoolExecutor(max_workers=workers) as exe:
            futures = {exe.submit(process_case, args): args[0]
                       for args in lst}
            for future in as_completed(futures):
                cid, slices = future.result()
                if not slices:
                    print(f"  skip {cid}: no DICOM")
                    continue
                for fname, img, msk in slices:
                    with h5py.File(os.path.join(out_dir, fname), "w") as f:
                        f.create_dataset("image", data=img, compression="gzip")
                        f.create_dataset("mask",  data=msk, compression="gzip")
                    cnt += 1
        print(f"  {split}: {cnt} slices -> {out_dir}")

# ── Run ──────────────────────────────────────────────────────────

NRRD_DIR = ("/content/drive/MyDrive/dataset /"
            "NCI-ISBI-2013-Prostate-Challenge-Training/Training")
DICOM_3T = "/content/drive/MyDrive/dataset /prostate_3t"
DICOM_DX = "/content/drive/MyDrive/dataset /prostate_diagnosis"

process_nci_site(DICOM_3T, NRRD_DIR, "Domain1", H5_BASE)  # Siemens
process_nci_site(DICOM_DX, NRRD_DIR, "Domain2", H5_BASE)  # Philips
print("NCI-ISBI done.")

Using device: cuda
Domain1: 30 matched cases
  train: 334 slices -> /content/data/Prostate_h5/Domain1/train
  test: 87 slices -> /content/data/Prostate_h5/Domain1/test
Domain2: 29 matched cases
  train: 282 slices -> /content/data/Prostate_h5/Domain2/train
  test: 87 slices -> /content/data/Prostate_h5/Domain2/test
NCI-ISBI done.


## Step 5 — Preprocess PROMISE12 (Domains 4, 5, 6)

| Domain | Cases | Scanner | Annotation |
|--------|-------|---------|------------|
| Domain4 | 00–11 | Siemens Avanto 1.5T (UCL) | keypoint |
| Domain5 | 12–22 | GE Signa 1.5T (BIDMC) | scribble\_noisy |
| Domain6 | 23–49 | Siemens Trio 3T (HK) | block (from bbox) |

In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed

PROMISE12_SPLITS = {
    "Domain4": list(range(0,  12)),
    "Domain5": list(range(12, 23)),
    "Domain6": list(range(23, 50)),
}

# ── Per-case worker ──────────────────────────────────────────────

def process_promise12_case(args):
    """Returns (cid, [(fname, img, msk), ...]) for one MHD case."""
    cid, img_path, seg_path = args
    vol = sitk.GetArrayFromImage(
              sitk.ReadImage(img_path)).astype(np.float32)
    msk = (sitk.GetArrayFromImage(
               sitk.ReadImage(seg_path)) > 0).astype(np.uint8)
    vol = normalize_vol_gpu(vol)
    results = []
    for i in range(vol.shape[0]):
        if msk[i].sum() == 0:
            continue
        results.append((
            f"Case{cid:02d}_s{i:03d}.h5",
            rs_img_gpu(vol[i]),
            rs_msk_gpu(msk[i]),
        ))
    return cid, results

# ── Main site processor ──────────────────────────────────────────

def process_promise12_site(mhd_base, domain, case_ids, out_base,
                            ratio=TRAIN_RATIO, workers=4):
    paired = []
    for cid in case_ids:
        img = os.path.join(mhd_base, f"Case{cid:02d}.mhd")
        seg = os.path.join(mhd_base, f"Case{cid:02d}_segmentation.mhd")
        if os.path.exists(img) and os.path.exists(seg):
            paired.append((cid, img, seg))
        else:
            print(f"  missing Case{cid:02d}")

    print(f"{domain}: {len(paired)} cases found")
    n_tr   = max(1, int(len(paired) * ratio))
    splits = {"train": paired[:n_tr], "test": paired[n_tr:]}

    for split, lst in splits.items():
        out_dir = os.path.join(out_base, domain, split)
        os.makedirs(out_dir, exist_ok=True)
        cnt = 0
        with ThreadPoolExecutor(max_workers=workers) as exe:
            futures = {exe.submit(process_promise12_case, args): args[0]
                       for args in lst}
            for future in as_completed(futures):
                cid, slices = future.result()
                for fname, img, msk in slices:
                    with h5py.File(os.path.join(out_dir, fname), "w") as f:
                        f.create_dataset("image", data=img, compression="gzip")
                        f.create_dataset("mask",  data=msk, compression="gzip")
                    cnt += 1
        print(f"  {split}: {cnt} slices -> {out_dir}")

# ── Run ──────────────────────────────────────────────────────────

MHD_BASE = "/content/drive/MyDrive/dataset /8026660/training_data"
for dom, cases in PROMISE12_SPLITS.items():
    process_promise12_site(MHD_BASE, dom, cases, H5_BASE)
print("PROMISE12 done.")

Domain4: 12 cases found
  train: 188 slices -> /content/data/Prostate_h5/Domain4/train
  test: 64 slices -> /content/data/Prostate_h5/Domain4/test
Domain5: 11 cases found
  train: 116 slices -> /content/data/Prostate_h5/Domain5/train
  test: 42 slices -> /content/data/Prostate_h5/Domain5/test
Domain6: 27 cases found
  train: 299 slices -> /content/data/Prostate_h5/Domain6/train
  test: 69 slices -> /content/data/Prostate_h5/Domain6/test
PROMISE12 done.


## Step 6 — Sparse annotation synthesis
Generates all annotation types for every train slice so the H5 schema is uniform across domains.

In [ ]:
# Re-upgrade numpy/scipy THEN hard restart.
# After restart, all cells run cleanly with no version conflicts.
import subprocess, sys, os, signal
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "numpy>=2.0", "scipy>=1.14.0"], check=True)
print("Packages upgraded. Restarting now...")
os.kill(os.getpid(), signal.SIGKILL)

In [2]:
# ── Install flwr with latest compatible deps ──────────────────────
!pip install -q flwr grpcio protobuf

# Force numpy/scipy back to 2.x (flwr latest supports it)
!pip install -q -U numpy scipy scikit-image

import flwr as fl, numpy as np, scipy
print(f"flwr    {fl.__version__}")
print(f"numpy   {np.__version__}")
print(f"scipy   {scipy.__version__}")

from flwr.common import Parameters, FitRes, EvaluateRes, \
    ndarrays_to_parameters, parameters_to_ndarrays
print("flwr symbols OK")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is

In [7]:
import glob
from skimage.morphology import skeletonize
from scipy.ndimage import gaussian_filter
import os
DOMAINS = ["Domain1","Domain2","Domain4","Domain5","Domain6"]


def gen_block(mask):
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    eroded = cv2.erode(mask, ker, iterations=1)
    out = np.full_like(mask, 2, dtype=np.uint8)  # 2=unlabeled
    out[mask   == 0] = 0
    out[eroded == 1] = 1
    return out


def gen_keypoint(mask):
    out = np.full_like(mask, 2, dtype=np.uint8)
    out[mask == 0] = 0
    ys, xs = np.where(mask == 1)
    if len(ys) == 0:
        return out
    cy, cx = int(ys.mean()), int(xs.mean())
    kp = np.zeros_like(mask, dtype=np.float32)
    kp[cy, cx] = 1.0
    kp = gaussian_filter(kp, sigma=3)
    out[kp > kp.max() * 0.3] = 1
    return out


def gen_scribble(mask, noisy=False):
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    eroded = cv2.erode(mask, ker, iterations=2)
    if eroded.sum() == 0:
        eroded = mask.copy()
    skel = skeletonize(eroded).astype(np.uint8)
    if noisy:
        skel = (skel * (np.random.rand(*skel.shape) > 0.4)).astype(np.uint8)
    out = np.full_like(mask, 2, dtype=np.uint8)
    out[mask == 0] = 0
    out[skel == 1] = 1
    return out


def gen_box(mask):
    """Bounding box preprocessed to block (paper Sec III-D)."""
    ys, xs = np.where(mask == 1)
    if len(ys) == 0:
        return np.zeros_like(mask, dtype=np.uint8)
    y1, y2, x1, x2 = ys.min(), ys.max(), xs.min(), xs.max()
    pad_y = max(1, int((y2 - y1) * 0.10))
    pad_x = max(1, int((x2 - x1) * 0.10))
    out = np.zeros_like(mask, dtype=np.uint8)
    out[y1+pad_y : y2-pad_y+1, x1+pad_x : x2-pad_x+1] = 1
    return out


for domain in DOMAINS:
    train_dir = os.path.join(H5_BASE, domain, "train")
    h5_files  = sorted(glob.glob(os.path.join(train_dir, "*.h5")))
    print(f"{domain}: annotating {len(h5_files)} slices...", end="", flush=True)
    for fpath in h5_files:
        with h5py.File(fpath, "a") as f:
            msk = f["mask"][:].astype(np.uint8)
            if "block"          not in f:
                f.create_dataset("block",
                    data=gen_block(msk),             compression="gzip")
            if "keypoint"       not in f:
                f.create_dataset("keypoint",
                    data=gen_keypoint(msk),          compression="gzip")
            if "scribble"       not in f:
                f.create_dataset("scribble",
                    data=gen_scribble(msk, False),   compression="gzip")
            if "scribble_noisy" not in f:
                f.create_dataset("scribble_noisy",
                    data=gen_scribble(msk, True),    compression="gzip")
            if "box"            not in f:
                f.create_dataset("box",
                    data=gen_box(msk),               compression="gzip")
    print(" done.")

print("Annotation synthesis complete.")


Domain1: annotating 334 slices... done.
Domain2: annotating 282 slices... done.
Domain4: annotating 188 slices... done.
Domain5: annotating 116 slices... done.
Domain6: annotating 299 slices... done.
Annotation synthesis complete.


## Step 7 — Verify H5 dataset structure

In [9]:
ANNOT_KEY = {
    "Domain1": "block",
    "Domain2": "keypoint",
    "Domain4": "keypoint",
    "Domain5": "scribble_noisy",
    "Domain6": "block",
}

print(f"{"Domain":<10} {"Split":<8} {"Slices":<8} Keys")
print("-" * 60)
for dom in DOMAINS:
    for split in ["train", "test"]:
        d = os.path.join(H5_BASE, dom, split)
        files = sorted(glob.glob(os.path.join(d, "*.h5")))
        if not files:
            print(f"{dom:<10} {split:<8} MISSING")
            continue
        with h5py.File(files[0], "r") as f:
            keys = list(f.keys())
        print(f"{dom:<10} {split:<8} {len(files):<8} {keys}")

# Deep check one sample
sample = glob.glob(f"{H5_BASE}/Domain1/train/*.h5")[0]
print(f"Sample: {os.path.basename(sample)}")
with h5py.File(sample, "r") as f:
    for k in f.keys():
        arr = f[k][:]
        print(f"  {k:<18} shape={arr.shape}  "
              f"dtype={arr.dtype}  unique={np.unique(arr)[:6]}")


Domain     Split    Slices   Keys
------------------------------------------------------------
Domain1    train    334      ['block', 'box', 'image', 'keypoint', 'mask', 'scribble', 'scribble_noisy']
Domain1    test     87       ['image', 'mask']
Domain2    train    282      ['block', 'box', 'image', 'keypoint', 'mask', 'scribble', 'scribble_noisy']
Domain2    test     87       ['image', 'mask']
Domain4    train    188      ['block', 'box', 'image', 'keypoint', 'mask', 'scribble', 'scribble_noisy']
Domain4    test     64       ['image', 'mask']
Domain5    train    116      ['block', 'box', 'image', 'keypoint', 'mask', 'scribble', 'scribble_noisy']
Domain5    test     42       ['image', 'mask']
Domain6    train    299      ['block', 'box', 'image', 'keypoint', 'mask', 'scribble', 'scribble_noisy']
Domain6    test     69       ['image', 'mask']
Sample: Prostate3T-01-0006_s017.h5
  block              shape=(384, 384)  dtype=uint8  unique=[0 1 2]
  box                shape=(384, 384)  dtyp

## Step 8 — Link data into FedLPPA expected path

In [10]:
os.chdir("/content/FedLPPA/code_v4")
os.makedirs("/content/FedLPPA/data", exist_ok=True)

DATA_LINK = "/content/FedLPPA/data/Prostate_h5"
if not os.path.exists(DATA_LINK):
    os.symlink(H5_BASE, DATA_LINK)
    print(f"Symlinked {H5_BASE} -> {DATA_LINK}")
else:
    print("Symlink already exists.")

for dom in DOMAINS:
    tr = len(os.listdir(os.path.join(DATA_LINK, dom, "train")))
    te = len(os.listdir(os.path.join(DATA_LINK, dom, "test")))
    print(f"  {dom}: {tr} train, {te} test")


Symlinked /content/data/Prostate_h5 -> /content/FedLPPA/data/Prostate_h5
  Domain1: 334 train, 87 test
  Domain2: 282 train, 87 test
  Domain4: 188 train, 64 test
  Domain5: 116 train, 42 test
  Domain6: 299 train, 69 test


## Step 9 — Run FedLPPA training

**Site mapping:**

| cid | client | Domain | sup_type | Description |
|-----|--------|--------|----------|-------------|
| 0 | client1 | Domain1 | block | NCI-ISBI Siemens |
| 1 | client2 | Domain2 | keypoint | NCI-ISBI Philips |
| 2 | client4 | Domain4 | keypoint | PROMISE12 Siemens-UCL |
| 3 | client5 | Domain5 | scribble_noisy | PROMISE12 GE-BIDMC |
| 4 | client6 | Domain6 | block | PROMISE12 Siemens-HK |

In [47]:
import subprocess, time, os
from tqdm.notebook import tqdm

os.chdir("/content/FedLPPA/code_v4")

SCRIPT    = "flower_pCE_2D_v4_FedLPPA.py"
PORT      = "8095"
ADDRESS   = f"127.0.0.1:{PORT}"
N_CLIENTS = 5

BASE = (
    f"python {SCRIPT} "
    f"--root_path ../data/Prostate_h5 "
    f"--num_classes 2 --in_chns 1 --img_class prostate "
    f"--exp prostate/FedLPPA "
    f"--model unet_univ5 "
    f"--max_iterations 25000 --iters 10 --eval_iters 10 --tsne_iters 500 "
    f"--batch_size 12 --base_lr 0.01 --amp 0 "
    f"--server_address {ADDRESS} "
    f"--strategy FedUniV2.1 "
    f"--min_num_clients {N_CLIENTS} "
    f"--img_size 384 "
    f"--alpha 0.1 --beta 0.5 "
    f"--prompt universal --attention dual "
    f"--dual_init aggregated --label_prompt 1 "
)

CLIENTS = [
    (0, "client1", "block"),
    (1, "client2", "keypoint"),
    (2, "client4", "keypoint"),
    (3, "client5", "scribble_noisy"),
    (4, "client6", "block"),
]

LOG_DIR = "/content/FedLPPA/logs/prostate"
os.makedirs(LOG_DIR, exist_ok=True)
for lf in os.listdir(LOG_DIR):
    open(os.path.join(LOG_DIR, lf), "w").close()
print("Logs cleared.")

# ── Step 1: start server, wait for gRPC ready ────────────────────
slf = open(f"{LOG_DIR}/server.log", "w")
srv = subprocess.Popen(
    BASE + "--role server --client client_all --sup_type mask --gpu 0",
    shell=True, stdout=slf, stderr=slf
)
print(f"Server PID={srv.pid} — waiting for gRPC...")

ready = False
with tqdm(total=60, desc="Server startup", unit="s", leave=True) as pbar:
    for _ in range(60):
        time.sleep(1)
        pbar.update(1)
        with open(f"{LOG_DIR}/server.log") as f:
            content = f.read()
        if "gRPC server running" in content:
            pbar.set_description("Server gRPC ready ✓")
            ready = True
            break
        if "Traceback" in content:
            pbar.set_description("Server crashed ✗")
            print("\nServer log:")
            print(content[-1500:])
            break

if not ready:
    print("Server did not become ready in 60s — check server.log")
    srv.terminate()
    raise RuntimeError("Server startup failed")

# ── Step 2: start clients with staggered launch ──────────────────
procs = []
print("\nLaunching clients...")
with tqdm(CLIENTS, desc="Clients", unit="client") as pbar:
    for cid, cname, sup in pbar:
        pbar.set_description(f"Starting {cname}")
        lf = open(f"{LOG_DIR}/{cname}.log", "w")
        p  = subprocess.Popen(
            BASE + f"--role client --cid {cid} --client {cname} "
                   f"--sup_type {sup} --gpu 0",
            shell=True, stdout=lf, stderr=lf
        )
        procs.append((p, cname, lf))
        time.sleep(2)  # stagger so clients don't all hit gRPC at once

# ── Step 3: wait for all clients to connect ──────────────────────
print("\nWaiting for all clients to connect...")
deadline = time.time() + 120
with tqdm(total=N_CLIENTS, desc="Clients connected", unit="client") as pbar:
    last_count = 0
    while time.time() < deadline:
        time.sleep(3)
        with open(f"{LOG_DIR}/server.log") as f:
            content = f.read()
        connected = content.count("registered")
        if connected > last_count:
            pbar.update(connected - last_count)
            last_count = connected
        if connected >= N_CLIENTS:
            break
        if "Traceback" in content:
            print("\nServer crashed after client connect:")
            print(content[-1500:])
            break

# ── Step 4: monitor training rounds ──────────────────────────────
print("\nTraining started — monitoring rounds...")
MAX_ROUNDS  = 25000
last_round  = 0
last_size   = 0

with tqdm(total=MAX_ROUNDS, desc="FL rounds", unit="round") as pbar:
    while srv.poll() is None:
        time.sleep(30)
        with open(f"{LOG_DIR}/server.log") as f:
            content = f.read()

        # Count completed rounds from log
        rounds_done = content.count("fit progress")
        if rounds_done > last_round:
            pbar.update(rounds_done - last_round)
            last_round = rounds_done

        # Print last meaningful server line
        lines = [l.strip() for l in content.splitlines()
                 if l.strip() and "DEBUG" not in l]
        if lines:
            pbar.set_postfix_str(lines[-1][-80:])

        # Detect crash
        if "Traceback" in content and last_round == 0:
            print("\nCrash detected:")
            print(content[-2000:])
            break

# ── Done ──────────────────────────────────────────────────────────
slf.close()
for p, name, lf in procs:
    p.terminate()
    lf.close()

print("\nTraining finished." if srv.returncode == 0
      else f"\nProcess exited with code {srv.returncode} — check server.log")

Logs cleared.
Server PID=35629 — waiting for gRPC...


Server startup:   0%|          | 0/60 [00:00<?, ?s/s]


Launching clients...


Clients:   0%|          | 0/5 [00:00<?, ?client/s]


Waiting for all clients to connect...


Clients connected:   0%|          | 0/5 [00:00<?, ?client/s]

KeyboardInterrupt: 

In [46]:
# ── Patch TreeEnergyLoss to be a no-op when tree_filter_cuda missing ──
fpath = "/content/FedLPPA/code_v4/flower_common_v4.py"

with open(fpath) as f:
    src = f.read()

# Find the TreeEnergyLoss __init__ and wrap the MST line
OLD = "        self.mst_layers = MinimumSpanningTree(TreeFilter2D.norm2_distance)"
NEW = """        if not TREE_FILTER_AVAILABLE or TreeFilter2D is None:
            self.mst_layers = None
            self._disabled = True
            return
        self._disabled = False
        self.mst_layers = MinimumSpanningTree(TreeFilter2D.norm2_distance)"""

src = src.replace(OLD, NEW)

# Also patch the forward method to return zero loss when disabled
OLD_FWD = "    def forward(self, prediction, image, low_feats, unlabeled_RoIs, epoch):"
NEW_FWD = """    def forward(self, prediction, image, low_feats, unlabeled_RoIs, epoch):
        if getattr(self, '_disabled', False):
            return prediction.sum() * 0.0"""

src = src.replace(OLD_FWD, NEW_FWD)

with open(fpath, "w") as f:
    f.write(src)
print("Patched TreeEnergyLoss in flower_common_v4.py")

# Apply same patch to flower_common_v4_addprostate.py
import shutil
shutil.copy(
    "/content/FedLPPA/code_v4/flower_common_v4.py",
    "/content/FedLPPA/code_v4/flower_common_v4_addprostate.py"
)
print("Copied patch to flower_common_v4_addprostate.py")

# Verify
with open(fpath) as f:
    lines = [l for l in f if "_disabled" in l]
print(f"Patch lines found: {len(lines)}")
print("Re-run training cell.")

Patched TreeEnergyLoss in flower_common_v4.py
Copied patch to flower_common_v4_addprostate.py
Patch lines found: 6
Re-run training cell.


In [48]:
import os, time
LOG_DIR = "/content/FedLPPA/logs/prostate"
time.sleep(5)

for name in ["server", "client1"]:
    fpath = f"{LOG_DIR}/{name}.log"
    size  = os.path.getsize(fpath)
    print(f"\n{'='*50}\n{name}.log  ({size} bytes)\n{'='*50}")
    with open(fpath) as f:
        print(f.read()[-2000:])


server.log  (5248 bytes)

INFO flwr 2026-05-29 18:33:50,556 | flower_common_v4.py:404 | round 10: fit failed

  0%|                             | 1/2500 [00:40<28:17:44, 40.76s/it]
Traceback (most recent call last):
  File "/content/FedLPPA/code_v4/flower_pCE_2D_v4_FedLPPA.py", line 742, in <module>
    main()
  File "/content/FedLPPA/code_v4/flower_pCE_2D_v4_FedLPPA.py", line 730, in main
    fl.server.start_server(
  File "/usr/local/lib/python3.12/dist-packages/flwr/server/app.py", line 176, in start_server
    hist = _fl(
           ^^^^
  File "/usr/local/lib/python3.12/dist-packages/flwr/server/app.py", line 217, in _fl
    hist = server.fit(num_rounds=config.num_rounds, timeout=config.round_timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/FedLPPA/code_v4/flower_common_v4.py", line 402, in fit
    res_fit = self.fit_round(server_round=current_round, timeout=timeout)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [49]:
# ════════════════════════════════════════════════════════════════
# FIX 1 — Disable GatedCRF loss (OOM on single GPU with 5 clients)
# Patch gate_crf_loss.py forward to return zero when out of memory
# ════════════════════════════════════════════════════════════════
fpath = "/content/FedLPPA/code_v4/utils/gate_crf_loss.py"
with open(fpath) as f:
    src = f.read()

OLD = "    def forward(self,"
NEW = """    def forward(self,"""

# Wrap entire forward in try/except OOM
OLD_FWD = "    def forward(self, predictions, scales, kernels):"
NEW_FWD = """    def forward(self, predictions, scales, kernels):
        try:
            pass  # replaced below
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            return predictions[0].sum() * 0.0
    def _forward_impl(self, predictions, scales, kernels):"""

# Simpler approach: just stub the whole class forward
stub = '''
    def forward(self, predictions, scales, kernels):
        """Stubbed — GatedCRF disabled (single-GPU multi-client OOM)."""
        torch.cuda.empty_cache()
        if isinstance(predictions, (list, tuple)):
            return predictions[0].sum() * 0.0
        return predictions.sum() * 0.0
'''

# Find forward method and replace up to next def
import re
src = re.sub(
    r'(    def forward\(self, predictions, scales, kernels\):.*?)(\n    def )',
    stub + r'\2',
    src,
    flags=re.DOTALL
)
with open(fpath, "w") as f:
    f.write(src)
print("Patched gate_crf_loss.py forward")

# ════════════════════════════════════════════════════════════════
# FIX 2 — Reduce per-client GPU memory footprint
# Set PYTORCH_CUDA_ALLOC_CONF and reduce batch size to 6
# ════════════════════════════════════════════════════════════════
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Set PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True")

# Verify stub
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-c",
     "import sys, torch; sys.path.insert(0,'.');"
     "from utils.gate_crf_loss import ModelLossSemsegGatedCRF;"
     "print('gate_crf_loss import OK')"],
    capture_output=True, text=True,
    cwd="/content/FedLPPA/code_v4"
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-1000:])

Patched gate_crf_loss.py forward
Set PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
gate_crf_loss import OK



In [69]:
import subprocess, time, os
from tqdm.notebook import tqdm

# ── Symlink model output to Drive ────────────────────────────────
MODEL_DIR  = "/content/FedLPPA/model/prostate/FedLPPA"
DRIVE_CKPT = "/content/drive/MyDrive/FedLPPA_prostate_ckpt"

os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs(os.path.dirname(MODEL_DIR), exist_ok=True)

if os.path.exists(MODEL_DIR) and not os.path.islink(MODEL_DIR):
    import shutil
    shutil.copytree(MODEL_DIR, DRIVE_CKPT, dirs_exist_ok=True)
    shutil.rmtree(MODEL_DIR)

if not os.path.exists(MODEL_DIR):
    os.symlink(DRIVE_CKPT, MODEL_DIR)
    print(f"Symlinked {MODEL_DIR} -> {DRIVE_CKPT}")
else:
    print("Checkpoint symlink already exists.")

# ── Setup ─────────────────────────────────────────────────────────
os.chdir("/content/FedLPPA/code_v4")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SCRIPT    = "flower_pCE_2D_v4_FedLPPA.py"
PORT      = "8095"
ADDRESS   = f"127.0.0.1:{PORT}"
N_CLIENTS = 5
LOG_DIR   = "/content/FedLPPA/logs/prostate"
MAX_ITER  = 10000

os.makedirs(LOG_DIR, exist_ok=True)
for lf in os.listdir(LOG_DIR):
    open(os.path.join(LOG_DIR, lf), "w").close()
print("Logs cleared.")

BASE = (
    f"python {SCRIPT} "
    f"--root_path ../data/Prostate_h5 "
    f"--num_classes 2 --in_chns 1 --img_class prostate "
    f"--exp prostate/FedLPPA "
    f"--model unet_univ5 "
    f"--max_iterations 3000 --iters 10 --eval_iters 10 --tsne_iters 100 "
    f"--batch_size 8 "        # back up from 6, amp handles memory
    f"--base_lr 0.003 "       # safe for fp16, still converges
    f"--amp 1 "               # mixed precision = fast
    f"--server_address {ADDRESS} "
    f"--strategy FedUniV2.1 "
    f"--min_num_clients {N_CLIENTS} "
    f"--img_size 384 "
    f"--alpha 0.1 --beta 0.5 "
    f"--prompt universal --attention dual "
    f"--dual_init aggregated --label_prompt 1 "
)
CLIENTS = [
    (0, "client1", "block"),
    (1, "client2", "keypoint"),
    (2, "client4", "keypoint"),
    (3, "client5", "scribble_noisy"),
    (4, "client6", "block"),
]

env = {**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}

# ── Start server ──────────────────────────────────────────────────
slf = open(f"{LOG_DIR}/server.log", "w")
srv = subprocess.Popen(
    BASE + "--role server --client client_all --sup_type mask --gpu 0",
    shell=True, stdout=slf, stderr=slf, env=env
)
print(f"Server PID={srv.pid}")

ready = False
with tqdm(total=180, desc="Server startup", unit="s") as pbar:
    for _ in range(180):
        time.sleep(1); pbar.update(1)
        content = open(f"{LOG_DIR}/server.log").read()
        if any(x in content for x in [
            "gRPC server running",
            "Flower ECE",
            "Requesting initial parameters",
            "Initializing global parameters",
        ]):
            pbar.set_description("Server ready ✓")
            ready = True
            break
        if "Traceback" in content:
            pbar.set_description("Server crashed ✗")
            print(content[-2000:])
            break

if not ready:
    srv.terminate()
    print("\nServer log:")
    print(open(f"{LOG_DIR}/server.log").read())
    raise RuntimeError("Server did not start in 180s")

# ── Start clients ─────────────────────────────────────────────────
procs = []
with tqdm(CLIENTS, desc="Launching clients", unit="client") as pbar:
    for cid, cname, sup in pbar:
        pbar.set_description(f"Starting {cname}")
        lf = open(f"{LOG_DIR}/{cname}.log", "w")
        p  = subprocess.Popen(
            BASE + f"--role client --cid {cid} --client {cname} "
                   f"--sup_type {sup} --gpu 0",
            shell=True, stdout=lf, stderr=lf, env=env
        )
        procs.append((p, cname, lf))
        time.sleep(3)

# ── Wait for clients to connect ───────────────────────────────────
print("\nWaiting for all clients to connect...")
deadline = time.time() + 180
with tqdm(total=N_CLIENTS, desc="Clients connected", unit="client") as pbar:
    last = 0
    while time.time() < deadline:
        time.sleep(3)
        # Count from CLIENT logs, not server log
        connected = 0
        for _, cname, _ in CLIENTS:
            try:
                with open(f"{LOG_DIR}/{cname}.log") as f:
                    clog = f.read()
                if any(x in clog for x in [
                    "ChannelConnectivity.READY",
                    "get_parameters",
                    "fit",
                    "Client",
                ]):
                    connected += 1
            except Exception:
                pass

        if connected > last:
            pbar.update(connected - last)
            last = connected
        if connected >= N_CLIENTS:
            pbar.set_description("All clients connected ✓")
            break

        # Also break if server already started round 1
        server_content = open(f"{LOG_DIR}/server.log").read()
        if "fit progress" in server_content or "round 1" in server_content.lower():
            pbar.update(N_CLIENTS - last)
            pbar.set_description("Training started ✓")
            break
        if "Traceback" in server_content:
            print("\nServer crashed:")
            print(server_content[-2000:])
            break

# ── Monitor training rounds ───────────────────────────────────────
FL_ROUNDS  = MAX_ITER // 10   # iters=10 per round
last_round = 0

print("\nTraining in progress...")
with tqdm(total=FL_ROUNDS, desc="FL rounds", unit="round") as pbar:
    while srv.poll() is None:
        time.sleep(30)
        content = open(f"{LOG_DIR}/server.log").read()

        done = content.count("fit progress")
        if done > last_round:
            pbar.update(done - last_round)
            last_round = done

        lines = [l.strip() for l in content.splitlines()
                 if l.strip() and "DEBUG" not in l and "WARNING" not in l]
        if lines:
            pbar.set_postfix_str(lines[-1][-80:])

        if "Traceback" in content and last_round == 0:
            print("\nCrash detected:")
            print(content[-2000:])
            break

# ── Cleanup ───────────────────────────────────────────────────────
slf.close()
for p, name, lf in procs:
    p.terminate(); lf.close()

if srv.returncode == 0 or last_round > 0:
    print(f"\nTraining complete. {last_round} rounds finished.")
    print(f"Checkpoints saved to: {DRIVE_CKPT}")
else:
    print(f"\nExited with code {srv.returncode} — check server.log")

Checkpoint symlink already exists.
Logs cleared.
Server PID=66728


Server startup:   0%|          | 0/180 [00:00<?, ?s/s]

Launching clients:   0%|          | 0/5 [00:00<?, ?client/s]


Waiting for all clients to connect...


Clients connected:   0%|          | 0/5 [00:00<?, ?client/s]


Training in progress...


FL rounds:   0%|          | 0/1000 [00:00<?, ?round/s]

KeyboardInterrupt: 

In [70]:
import subprocess, os

LOG_DIR = "/content/FedLPPA/logs/prostate"

# Are processes alive?
r = subprocess.run(["ps", "aux"], capture_output=True, text=True)
flower = [l for l in r.stdout.splitlines() if "flower_pCE" in l]
print(f"Live processes: {len(flower)}")

# Full server log
with open(f"{LOG_DIR}/server.log") as f:
    content = f.read()
print(f"\nServer log ({len(content)} bytes), last 20 lines:")
lines = [l for l in content.splitlines() if l.strip()]
for l in lines[-20:]:
    print(" ", l)

# Client1 last lines
print("\nClient1 last 10 lines:")
with open(f"{LOG_DIR}/client1.log") as f:
    clines = [l for l in f.read().splitlines() if l.strip()]
for l in clines[-10:]:
    print(" ", l)

Live processes: 0

Server log (5452 bytes), last 20 lines:
    File "/content/FedLPPA/code_v4/flower_common_v4.py", line 402, in fit
      res_fit = self.fit_round(server_round=current_round, timeout=timeout)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    File "/usr/local/lib/python3.12/dist-packages/flwr/server/server.py", line 209, in fit_round
      client_instructions = self.strategy.configure_fit(
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    File "/usr/local/lib/python3.12/dist-packages/flwr/server/strategy/fedavg.py", line 184, in configure_fit
      clients = client_manager.sample(
                ^^^^^^^^^^^^^^^^^^^^^^
    File "/usr/local/lib/python3.12/dist-packages/flwr/server/client_manager.py", line 180, in sample
      self.wait_for(min_num_clients)
    File "/usr/local/lib/python3.12/dist-packages/flwr/server/client_manager.py", line 125, in wait_for
      return self._cv.wait_for(
             ^^^^^^^^^^^^^^^^^^
    File 

In [71]:
import subprocess, time, os
from tqdm.notebook import tqdm

os.chdir("/content/FedLPPA/code_v4")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SCRIPT    = "flower_pCE_2D_v4_FedLPPA.py"
ADDRESS   = "127.0.0.1:8095"
N_CLIENTS = 5
LOG_DIR   = "/content/FedLPPA/logs/prostate"
MAX_ITER  = 3000

os.makedirs(LOG_DIR, exist_ok=True)
for lf in os.listdir(LOG_DIR):
    open(os.path.join(LOG_DIR, lf), "w").close()
print("Logs cleared.")

BASE = (
    f"python {SCRIPT} "
    f"--root_path ../data/Prostate_h5 "
    f"--num_classes 2 --in_chns 1 --img_class prostate "
    f"--exp prostate/FedLPPA "
    f"--model unet_univ5 "
    f"--max_iterations {MAX_ITER} --iters 10 --eval_iters 10 --tsne_iters 100 "
    f"--batch_size 8 --base_lr 0.003 --amp 1 "
    f"--server_address {ADDRESS} "
    f"--strategy FedUniV2.1 --min_num_clients {N_CLIENTS} "
    f"--img_size 384 --alpha 0.1 --beta 0.5 "
    f"--prompt universal --attention dual "
    f"--dual_init aggregated --label_prompt 1 "
)

CLIENTS = [
    (0, "client1", "block"),
    (1, "client2", "keypoint"),
    (2, "client4", "keypoint"),
    (3, "client5", "scribble_noisy"),
    (4, "client6", "block"),
]

env = {**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}

# ── Start server ──────────────────────────────────────────────────
open(f"{LOG_DIR}/server.log", "w").close()
srv = subprocess.Popen(
    BASE + "--role server --client client_all --sup_type mask --gpu 0",
    shell=True,
    stdout=open(f"{LOG_DIR}/server.log", "w"),
    stderr=subprocess.STDOUT,
    env=env
)
print(f"Server PID={srv.pid}")

# Wait for gRPC ready
ready = False
with tqdm(total=180, desc="Server startup", unit="s") as pbar:
    for _ in range(180):
        time.sleep(1); pbar.update(1)
        c = open(f"{LOG_DIR}/server.log").read()
        if any(x in c for x in ["gRPC server running","Flower ECE",
                                  "Requesting initial parameters",
                                  "Initializing global parameters"]):
            pbar.set_description("Server ready ✓")
            ready = True; break
        if "Traceback" in c:
            pbar.set_description("Crashed ✗")
            print(c[-1500:]); break

if not ready:
    srv.terminate()
    raise RuntimeError("Server did not start")

# ── Start clients ─────────────────────────────────────────────────
for cid, cname, sup in CLIENTS:
    subprocess.Popen(
        BASE + f"--role client --cid {cid} --client {cname} "
               f"--sup_type {sup} --gpu 0",
        shell=True,
        stdout=open(f"{LOG_DIR}/{cname}.log", "w"),
        stderr=subprocess.STDOUT,
        env=env
    )
    print(f"Client {cname} started")
    time.sleep(3)

# ── Cell ends here — training runs in background ──────────────────
print("\nAll processes launched. Cell is now FREE.")
print("Run the monitor cell below to track progress.")
print("Do NOT re-run this cell unless you want to restart training.")

Logs cleared.
Server PID=69680


Server startup:   0%|          | 0/180 [00:00<?, ?s/s]

Client client1 started
Client client2 started
Client client4 started
Client client5 started
Client client6 started

All processes launched. Cell is now FREE.
Run the monitor cell below to track progress.
Do NOT re-run this cell unless you want to restart training.


In [73]:
import subprocess, os, re

LOG_DIR = "/content/FedLPPA/logs/prostate"

r = subprocess.run(["ps", "aux"], capture_output=True, text=True)
alive = [l for l in r.stdout.splitlines() if "flower_pCE" in l]
print(f"Live processes: {len(alive)}")

with open(f"{LOG_DIR}/server.log") as f:
    content = f.read()

tqdm_lines = [l for l in content.splitlines() if "/300 [" in l]
if tqdm_lines:
    print(f"Last progress: {tqdm_lines[-1].strip()}")

print("\nLast 15 server lines:")
lines = [l for l in content.splitlines() if l.strip()]
for l in lines[-15:]:
    print(" ", l)

print("\nClient1 last 5 lines:")
with open(f"{LOG_DIR}/client1.log") as f:
    clines = [l for l in f.read().splitlines() if l.strip()]
for l in clines[-5:]:
    print(" ", l)

Live processes: 10
Last progress: 0%|                                 | 1/300 [00:10<51:34, 10.35s/it]

Last 15 server lines:
  INFO flwr 2026-05-29 19:05:02,724 | server.py:277 | Received initial parameters from one random client
  INFO flwr 2026-05-29 19:05:02,725 | flower_common_v4.py:319 | Evaluating initial parameters
  /content/FedLPPA/code_v4/val_2D.py:63: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
    with autocast(enabled=amp):
  INFO flwr 2026-05-29 19:05:10,604 | flower_common_v4.py:323 | initial parameters (loss, other metrics): 0.0, {'val_1_dice': np.float64(0.0), 'val_1_hd95': np.float64(0.0), 'val_1_recall': np.float64(0.0), 'val_1_precision': np.float64(0.0), 'val_1_jc': np.float64(0.0), 'val_1_specificity': np.float64(0.0), 'val_1_ravd': np.float64(0.0), 'val_mean_dice': np.float64(0.0), 'val_mean_hd95': np.float64(0.0), 'val_mean_recall': np.float64(0.0), 'val_mean_precision': np.float64(0

In [74]:
fpath = "/content/FedLPPA/code_v4/flower_pCE_2D_v4_FedLPPA.py"
with open(fpath) as f:
    src = f.read()

# Find the loss computation and add NaN guard before backward
OLD = "                scaler.scale(loss).backward()"
NEW = """                # Guard against NaN loss (Dice on empty pseudo-labels)
                if not torch.isfinite(loss):
                    loss = loss_ce  # fallback to CE only
                scaler.scale(loss).backward()"""

if OLD in src:
    src = src.replace(OLD, NEW)
    with open(fpath, "w") as f:
        f.write(src)
    print("NaN guard patched.")
else:
    # Find the actual loss backward line
    for i, l in enumerate(src.splitlines()):
        if "backward" in l and "loss" in l:
            print(f"line {i}: {l}")

NaN guard patched.


In [79]:
with open("/content/FedLPPA/code_v4/flower_pCE_2D_v4_FedLPPA.py") as f:
    lines = f.readlines()
print("".join(lines[228:380]))

                        loss_meta = torch.tensor(0.0)
                        lam = 0.0

                if (self.args.strategy in ['FedAP', 'FedAPLC'] and (self.current_iter + 1) == self.args.iters) \
                    or (self.args.strategy == 'MetaFed' and (self.current_iter + 1) == (self.args.common_iters + self.args.iters)):
                    print(self.args.cid, 'get_bn_stats', self.current_iter + 1)
                    bnm, bnv = get_bn_stats(self.args, self.model.model, self.trainloader)

                if self.args.strategy in ['FedLC', 'FedALALC', 'FedAPLC']:
                    loss_lc = 0
                    for other_client in range(self.args.min_num_clients):
                        if other_client == self.args.cid:
                           continue 
                        with torch.no_grad():
                            _heatmaps = self.model(volume_batch, other_client)[-2]
                            # print(heatmaps[-1].shape, _heatmaps[-1].shape)
            

In [80]:
fpath = "/content/FedLPPA/code_v4/flower_pCE_2D_v4_FedLPPA.py"
with open(fpath) as f:
    src = f.read()

OLD = """                if self.args.strategy in ['FedUniV2', 'FedUniV2.1']:
                    loss_uni = torch.tensor(0.0).cuda()
                    out_gatedcrf = gatecrf_loss(
                      outputs_soft,
                      loss_gatedcrf_kernels_desc,
                      loss_gatedcrf_radius,
                      volume_batch,
                      self.args.img_size,
                      self.args.img_size
                  )["loss"]
                    loss = loss + 0.1 * out_gatedcrf
                    # MSE, KL
                    for other_client in range(self.args.min_num_clients):
                        if other_client == self.args.cid:
                           continue
                        # print(fuse_feature.shape, prompts.shape)
                        # loss_uni += mse_loss(prompts[:, self.args.cid], prompts[:, other_client].detach())
                        loss_uni += kl_loss(prompts[self.args.cid], prompts[other_client].detach())
                    loss_uni = -loss_uni / (self.args.min_num_clients - 1)
                    # loss_uni = -torch.log(loss_uni / (self.args.min_num_clients - 1))
                    loss = torch.add(loss, loss_uni, alpha=self.args.alpha)
                    # dual branches
                    outputs_soft_auxiliary = torch.softmax(outputs_auxiliary, dim=1)
                    # pseudo_alpha = 0.75 # np.random.random()
                    pseudo_alpha = np.random.uniform(0, 1)
                    pseudo_label = pseudo_alpha * outputs_soft.detach() + (1 - pseudo_alpha) * outputs_soft_auxiliary.detach()
                    pseudo_label = torch.argmax(pseudo_label, dim=1)
                    # print(self.cid, pseudo_alpha, torch.unique(pseudo_label))
                    loss_pls_1 = dice_loss(outputs_soft, pseudo_label.unsqueeze(1))
                    loss_pls_2 = dice_loss(outputs_soft_auxiliary, pseudo_label.unsqueeze(1))
                    loss_pls = (loss_pls_1 + loss_pls_2) / 2
                    loss = torch.add(loss, loss_pls, alpha=self.args.beta)"""

NEW = """                if self.args.strategy in ['FedUniV2', 'FedUniV2.1']:
                    loss_uni = torch.tensor(0.0).cuda()
                    # GatedCRF — guard against NaN
                    out_gatedcrf = gatecrf_loss(
                      outputs_soft,
                      loss_gatedcrf_kernels_desc,
                      loss_gatedcrf_radius,
                      volume_batch,
                      self.args.img_size,
                      self.args.img_size
                  )["loss"]
                    if torch.isfinite(out_gatedcrf):
                        loss = loss + 0.1 * out_gatedcrf
                    # KL prompt loss — guard against NaN
                    for other_client in range(self.args.min_num_clients):
                        if other_client == self.args.cid:
                           continue
                        kl = kl_loss(prompts[self.args.cid], prompts[other_client].detach())
                        if torch.isfinite(kl):
                            loss_uni += kl
                    if self.args.min_num_clients > 1:
                        loss_uni = -loss_uni / (self.args.min_num_clients - 1)
                    if torch.isfinite(loss_uni):
                        loss = torch.add(loss, loss_uni, alpha=self.args.alpha)
                    # Pseudo-label Dice loss — guard against NaN (all-zero pseudo labels)
                    outputs_soft_auxiliary = torch.softmax(outputs_auxiliary, dim=1)
                    pseudo_alpha = np.random.uniform(0, 1)
                    pseudo_label = pseudo_alpha * outputs_soft.detach() + (1 - pseudo_alpha) * outputs_soft_auxiliary.detach()
                    pseudo_label = torch.argmax(pseudo_label, dim=1)
                    loss_pls_1 = dice_loss(outputs_soft, pseudo_label.unsqueeze(1))
                    loss_pls_2 = dice_loss(outputs_soft_auxiliary, pseudo_label.unsqueeze(1))
                    loss_pls = (loss_pls_1 + loss_pls_2) / 2
                    if torch.isfinite(loss_pls):
                        loss = torch.add(loss, loss_pls, alpha=self.args.beta)"""

if OLD in src:
    src = src.replace(OLD, NEW)
    with open(fpath, "w") as f:
        f.write(src)
    print("NaN guards applied to all 3 FedUniV2.1 loss terms.")
else:
    print("Pattern not found exactly — check indentation in the file.")

NaN guards applied to all 3 FedUniV2.1 loss terms.


In [82]:
import subprocess, time, os, re
from tqdm.notebook import tqdm

os.chdir("/content/FedLPPA/code_v4")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SCRIPT    = "flower_pCE_2D_v4_FedLPPA.py"
ADDRESS   = "127.0.0.1:8095"
N_CLIENTS = 5
LOG_DIR   = "/content/FedLPPA/logs/prostate"
MAX_ITER  = 3000
FL_ROUNDS = MAX_ITER // 10

os.makedirs(LOG_DIR, exist_ok=True)
for lf in os.listdir(LOG_DIR):
    open(os.path.join(LOG_DIR, lf), "w").close()
print("Logs cleared.")

BASE = (
    f"python {SCRIPT} "
    f"--root_path ../data/Prostate_h5 "
    f"--num_classes 2 --in_chns 1 --img_class prostate "
    f"--exp prostate/FedLPPA --model unet_univ5 "
    f"--max_iterations {MAX_ITER} --iters 10 --eval_iters 10 --tsne_iters 100 "
    f"--batch_size 8 --base_lr 0.003 --amp 1 "
    f"--server_address {ADDRESS} "
    f"--strategy FedUniV2.1 --min_num_clients {N_CLIENTS} "
    f"--img_size 384 --alpha 0.1 --beta 0.5 "
    f"--prompt universal --attention dual "
    f"--dual_init aggregated --label_prompt 1 "
)

CLIENTS = [
    (0, "client1", "block"),
    (1, "client2", "keypoint"),
    (2, "client4", "keypoint"),
    (3, "client5", "scribble_noisy"),
    (4, "client6", "block"),
]

env = {**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}

# ── Launch server in its own session (immune to cell SIGINT) ──────
open(f"{LOG_DIR}/server.log", "w").close()
srv = subprocess.Popen(
    BASE + "--role server --client client_all --sup_type mask --gpu 0",
    shell=True,
    stdout=open(f"{LOG_DIR}/server.log", "w"),
    stderr=subprocess.STDOUT,
    env=env,
    preexec_fn=os.setsid   # <-- own process group, survives cell stop
)
print(f"Server PID={srv.pid}")

# ── Wait for gRPC ─────────────────────────────────────────────────
ready = False
with tqdm(total=180, desc="Server startup", unit="s") as pbar:
    for _ in range(180):
        time.sleep(1); pbar.update(1)
        c = open(f"{LOG_DIR}/server.log").read()
        if any(x in c for x in ["gRPC server running", "Flower ECE",
                                  "Requesting initial parameters",
                                  "Initializing global parameters"]):
            pbar.set_description("Server ready ✓"); ready = True; break
        if "Traceback" in c:
            pbar.set_description("Crashed ✗"); print(c[-1500:]); break

if not ready:
    raise RuntimeError("Server failed — check server.log")

# ── Launch clients in their own sessions ──────────────────────────
for cid, cname, sup in CLIENTS:
    subprocess.Popen(
        BASE + f"--role client --cid {cid} --client {cname} "
               f"--sup_type {sup} --gpu 0",
        shell=True,
        stdout=open(f"{LOG_DIR}/{cname}.log", "w"),
        stderr=subprocess.STDOUT,
        env=env,
        preexec_fn=os.setsid   # <-- own process group
    )
    print(f"  Client {cname} started")
    time.sleep(3)

# ── Inline live monitor (stopping cell won't kill training) ───────
print("\nMonitoring — you can safely stop this cell, training continues.\n")
last_round = 0
no_progress_count = 0

with tqdm(total=FL_ROUNDS, desc="FL rounds", unit="round") as pbar:
    while True:
        time.sleep(20)

        # Check if server process is still alive
        r = subprocess.run(["ps", "aux"], capture_output=True, text=True)
        alive = sum(1 for l in r.stdout.splitlines() if "flower_pCE" in l)

        content = open(f"{LOG_DIR}/server.log").read()

        # Round count from inner tqdm
        tqdm_lines = [l for l in content.splitlines() if f"/{FL_ROUNDS} [" in l]
        rounds_now = 0
        eta = "?"
        if tqdm_lines:
            m = re.search(rf'(\d+)/{FL_ROUNDS} \[.*?<([\d:]+)', tqdm_lines[-1])
            if m:
                rounds_now = int(m.group(1))
                eta = m.group(2)

        if rounds_now > last_round:
            pbar.update(rounds_now - last_round)
            last_round = rounds_now
            no_progress_count = 0
        else:
            no_progress_count += 1

        # Latest metrics
        mlines = [l for l in content.splitlines() if "mean_dice" in l]
        dice_str = "0.0000"
        if mlines:
            m = re.search(r'mean_dice : ([\d.]+)', mlines[-1])
            if m:
                dice_str = m.group(1)

        pbar.set_postfix({
            "dice":  dice_str,
            "ETA":   eta,
            "procs": alive,
        })

        # Detect training finished
        if rounds_now >= FL_ROUNDS:
            pbar.set_description("Training complete ✓")
            break

        # Detect all processes dead
        if alive == 0:
            print(f"\nAll processes died at round {last_round}.")
            print("Training processes are detached — check logs:")
            print(f"  tail -20 {LOG_DIR}/server.log")
            break

        # Detect stall (no progress for 10 checks = ~3 min)
        if no_progress_count >= 10 and last_round == 0:
            print("\nNo progress detected — possible crash. Check:")
            lines = [l for l in content.splitlines() if l.strip()]
            for l in lines[-10:]: print(" ", l)
            break

print(f"\nDone. Rounds: {last_round}/{FL_ROUNDS} | Checkpoints: /content/drive/MyDrive/FedLPPA_prostate_ckpt")

Logs cleared.
Server PID=72711


Server startup:   0%|          | 0/180 [00:00<?, ?s/s]

  Client client1 started
  Client client2 started
  Client client4 started
  Client client5 started
  Client client6 started

Monitoring — you can safely stop this cell, training continues.



FL rounds:   0%|          | 0/300 [00:00<?, ?round/s]

KeyboardInterrupt: 

In [83]:
import subprocess, re, os

LOG_DIR = "/content/FedLPPA/logs/prostate"

r = subprocess.run(["ps", "aux"], capture_output=True, text=True)
alive = [l for l in r.stdout.splitlines() if "flower_pCE" in l]
print(f"Alive: {len(alive)}/6")

with open(f"{LOG_DIR}/server.log") as f:
    content = f.read()

tqdm_lines = [l for l in content.splitlines() if "/300 [" in l]
print(f"Progress lines: {len(tqdm_lines)}")
if tqdm_lines:
    print(f"Latest: {tqdm_lines[-1].strip()}")

mlines = [l for l in content.splitlines() if "mean_dice" in l]
if mlines:
    print(f"Metrics: {mlines[-1].strip()}")

print(f"\nLast 5 server lines:")
lines = [l for l in content.splitlines() if l.strip() and "DEBUG" not in l]
for l in lines[-5:]:
    print(f"  {l}")

Alive: 10/6
Progress lines: 2
Latest: 0%|                                 | 1/300 [00:10<54:40, 10.97s/it]
Metrics: (0.0, {'val_1_dice': np.float64(0.0), 'val_1_hd95': np.float64(0.0), 'val_1_recall': np.float64(0.0), 'val_1_precision': np.float64(0.0), 'val_1_jc': np.float64(0.0), 'val_1_specificity': np.float64(0.0), 'val_1_ravd': np.float64(0.0), 'val_mean_dice': np.float64(0.0), 'val_mean_hd95': np.float64(0.0), 'val_mean_recall': np.float64(0.0), 'val_mean_precision': np.float64(0.0), 'val_mean_jc': np.float64(0.0), 'val_mean_specificity': np.float64(0.0), 'val_mean_ravd': np.float64(0.0)})

Last 5 server lines:
  unet_univ5 parameters: 2.90M
  client 0 local_keys: ['decoder.out_conv.weight', 'decoder.out_conv.bias']
  (0.0, {'val_1_dice': np.float64(0.0), 'val_1_hd95': np.float64(0.0), 'val_1_recall': np.float64(0.0), 'val_1_precision': np.float64(0.0), 'val_1_jc': np.float64(0.0), 'val_1_specificity': np.float64(0.0), 'val_1_ravd': np.float64(0.0), 'val_mean_dice': np.float64(0.

In [87]:
import subprocess, time, os, re
from tqdm.notebook import tqdm

subprocess.run("pkill -f flower_pCE_2D_v4_FedLPPA", shell=True)
time.sleep(3)
print("Old processes killed.")

os.chdir("/content/FedLPPA/code_v4")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"

SCRIPT    = "flower_pCE_2D_v4_FedLPPA.py"
ADDRESS   = "127.0.0.1:8095"
N_CLIENTS = 5
LOG_DIR   = "/content/FedLPPA/logs/prostate"
MAX_ITER  = 3000
FL_ROUNDS = MAX_ITER // 10

os.makedirs(LOG_DIR, exist_ok=True)
for lf in os.listdir(LOG_DIR):
    open(os.path.join(LOG_DIR, lf), "w").close()
print("Logs cleared.")

BASE = (
    f"python {SCRIPT} "
    f"--root_path ../data/Prostate_h5 "
    f"--num_classes 2 --in_chns 1 --img_class prostate "
    f"--exp prostate/FedLPPA --model unet_univ5 "
    f"--max_iterations {MAX_ITER} --iters 10 --eval_iters 10 --tsne_iters 100 "
    f"--batch_size 4 "        # reduced from 8
    f"--base_lr 0.003 --amp 1 "
    f"--server_address {ADDRESS} "
    f"--strategy FedUniV2.1 --min_num_clients {N_CLIENTS} "
    f"--img_size 384 --alpha 0.1 --beta 0.5 "
    f"--prompt universal --attention dual "
    f"--dual_init aggregated --label_prompt 1 "
)

CLIENTS = [
    (0, "client1", "block"),
    (1, "client2", "keypoint"),
    (2, "client4", "keypoint"),
    (3, "client5", "scribble_noisy"),
    (4, "client6", "block"),
]

env = {**os.environ,
       "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:512"}

open(f"{LOG_DIR}/server.log", "w").close()
srv = subprocess.Popen(
    BASE + "--role server --client client_all --sup_type mask --gpu 0",
    shell=True,
    stdout=open(f"{LOG_DIR}/server.log", "w"),
    stderr=subprocess.STDOUT,
    env=env, preexec_fn=os.setsid
)
print(f"Server PID={srv.pid}")

ready = False
with tqdm(total=180, desc="Server startup", unit="s") as pbar:
    for _ in range(180):
        time.sleep(1); pbar.update(1)
        c = open(f"{LOG_DIR}/server.log").read()
        if any(x in c for x in ["gRPC server running","Flower ECE",
                                  "Requesting initial parameters",
                                  "Initializing global parameters"]):
            pbar.set_description("Server ready ✓"); ready = True; break
        if "Traceback" in c:
            pbar.set_description("Crashed ✗"); print(c[-1500:]); break

if not ready:
    raise RuntimeError("Server failed to start")

for cid, cname, sup in CLIENTS:
    subprocess.Popen(
        BASE + f"--role client --cid {cid} --client {cname} "
               f"--sup_type {sup} --gpu 0",
        shell=True,
        stdout=open(f"{LOG_DIR}/{cname}.log", "w"),
        stderr=subprocess.STDOUT,
        env=env, preexec_fn=os.setsid
    )
    print(f"  Client {cname} started")
    time.sleep(5)   # extra stagger so clients don't all allocate at once

print("\nTraining running — updates every 20s\n")
last_round = 0
no_progress = 0

with tqdm(total=FL_ROUNDS, desc="FL rounds", unit="round") as pbar:
    while True:
        time.sleep(20)
        content = open(f"{LOG_DIR}/server.log").read()

        tqdm_lines = [l for l in content.splitlines() if f"/{FL_ROUNDS} [" in l]
        rounds, eta = last_round, "?"
        if tqdm_lines:
            m = re.search(rf'(\d+)/{FL_ROUNDS} \[.*?<([\d:]+)', tqdm_lines[-1])
            if m:
                rounds = int(m.group(1))
                eta    = m.group(2)

        if rounds > last_round:
            pbar.update(rounds - last_round)
            last_round = rounds
            no_progress = 0
        else:
            no_progress += 1

        mlines = [l for l in content.splitlines()
                  if "mean_dice" in l and "iteration" in l]
        dice = "0.0000"
        if mlines:
            m = re.search(r'mean_dice : ([\d.]+)', mlines[-1])
            if m: dice = m.group(1)

        loss_str = ""
        try:
            with open(f"{LOG_DIR}/client1.log") as f:
                clines = [l for l in f.read().splitlines() if "iteration" in l]
            if clines:
                m = re.search(r'loss : ([\d.naf]+)', clines[-1])
                if m: loss_str = m.group(1)
        except Exception:
            pass

        r = subprocess.run(["ps","aux"], capture_output=True, text=True)
        alive = sum(1 for l in r.stdout.splitlines() if "flower_pCE" in l)

        pbar.set_postfix({"dice": dice, "ETA": eta,
                          "procs": alive, "loss": loss_str})

        # OOM check
        oom_clients = []
        for cname in ["client1","client2","client4","client5","client6"]:
            try:
                with open(f"{LOG_DIR}/{cname}.log") as f:
                    if "OutOfMemoryError" in f.read():
                        oom_clients.append(cname)
            except Exception:
                pass
        if oom_clients:
            print(f"\nOOM detected in: {oom_clients} — reduce batch_size further")
            break

        if rounds >= FL_ROUNDS:
            pbar.set_description("Complete ✓"); break
        if alive == 0:
            print("\nAll processes died"); break

Old processes killed.
Logs cleared.
Server PID=83877


Server startup:   0%|          | 0/180 [00:00<?, ?s/s]

  Client client1 started
  Client client2 started
  Client client4 started
  Client client5 started
  Client client6 started

Training running — updates every 20s



FL rounds:   0%|          | 0/300 [00:00<?, ?round/s]

KeyboardInterrupt: 

In [90]:
# ── Patch DataLoader to num_workers=0 ────────────────────────────
fpath = "/content/FedLPPA/code_v4/flower_pCE_2D_v4_FedLPPA.py"
with open(fpath) as f:
    src = f.read()

# Fix all DataLoader calls
import re
patched = re.sub(
    r'DataLoader\(([^)]+),\s*num_workers=\d+',
    lambda m: m.group(0).replace(
        f"num_workers={re.search(r'num_workers=(\d+)', m.group(0)).group(1)}",
        "num_workers=0"
    ),
    src
)

if patched != src:
    with open(fpath, "w") as f:
        f.write(patched)
    print("Patched: num_workers=0 in all DataLoaders")
else:
    # Manual check
    for i, l in enumerate(src.splitlines()):
        if "DataLoader" in l and "num_workers" in l:
            print(f"line {i}: {l.strip()}")

Patched: num_workers=0 in all DataLoaders


In [98]:
import subprocess, time, os, re
from IPython.display import clear_output

subprocess.run("pkill -f flower_pCE_2D_v4_FedLPPA", shell=True)
time.sleep(3)

os.chdir("/content/FedLPPA/code_v4")
env = {**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}

SCRIPT    = "flower_pCE_2D_v4_FedLPPA.py"
ADDRESS   = "127.0.0.1:8095"
N_CLIENTS = 5
LOG_DIR   = "/content/FedLPPA/logs/prostate"
MAX_ITER  = 3000
FL_ROUNDS = MAX_ITER // 10

os.makedirs(LOG_DIR, exist_ok=True)
for lf in os.listdir(LOG_DIR):
    open(os.path.join(LOG_DIR, lf), "w").close()

BASE = (
    f"python {SCRIPT} "
    f"--root_path ../data/Prostate_h5 "
    f"--num_classes 2 --in_chns 1 --img_class prostate "
    f"--exp prostate/FedLPPA --model unet_univ5 "
    f"--max_iterations {MAX_ITER} --iters 10 --eval_iters 10 --tsne_iters 100 "
    f"--batch_size 6 --base_lr 0.003 --amp 1 "
    f"--server_address {ADDRESS} "
    f"--strategy FedUniV2.1 --min_num_clients {N_CLIENTS} "
    f"--img_size 384 --alpha 0.1 --beta 0.5 "
    f"--prompt universal --attention dual "
    f"--dual_init aggregated --label_prompt 1 "
)

CLIENTS = [
    (0, "client1", "block"),
    (1, "client2", "keypoint"),
    (2, "client4", "keypoint"),
    (3, "client5", "scribble_noisy"),
    (4, "client6", "block"),
]

open(f"{LOG_DIR}/server.log", "w").close()
srv = subprocess.Popen(
    BASE + "--role server --client client_all --sup_type mask --gpu 0",
    shell=True,
    stdout=open(f"{LOG_DIR}/server.log", "w"),
    stderr=subprocess.STDOUT,
    env=env, preexec_fn=os.setsid
)

print("Server start ho raha hai...")
for _ in range(180):
    time.sleep(1)
    c = open(f"{LOG_DIR}/server.log").read()
    if any(x in c for x in ["gRPC server running", "Flower ECE",
                              "Requesting initial parameters",
                              "Initializing global parameters"]):
        print(f"Server ready! PID={srv.pid}")
        break
    if "Traceback" in c:
        print(c[-1000:])
        raise RuntimeError("Server failed")

for cid, cname, sup in CLIENTS:
    subprocess.Popen(
        BASE + f"--role client --cid {cid} --client {cname} "
               f"--sup_type {sup} --gpu 0",
        shell=True,
        stdout=open(f"{LOG_DIR}/{cname}.log", "w"),
        stderr=subprocess.STDOUT,
        env=env, preexec_fn=os.setsid
    )
    time.sleep(5)
print("Sab clients start!\n")

history = []

while True:
    time.sleep(10)
    try:
        with open(f"{LOG_DIR}/server.log") as f:
            content = f.read()

        # Round count from fit_round lines
        fit_lines     = [l for l in content.splitlines() if "fit_round" in l]
        current_round = len(fit_lines)

        # Dice from server eval lines
        mlines = [l for l in content.splitlines()
                  if "mean_dice" in l and "iteration" in l]
        dice = 0.0
        if mlines:
            m = re.search(r'mean_dice : ([\d.]+)', mlines[-1])
            if m: dice = float(m.group(1))

        # Loss — parse only pure iteration lines, skip False/tensor lines
        loss = 0.0
        with open(f"{LOG_DIR}/client1.log") as f:
            raw = f.read()
        # Only lines that start with INFO and contain "iteration"
        iter_lines = re.findall(
            r'INFO.*?iteration \d+ : lr: [\d.]+, loss : ([\d.]+)',
            raw
        )
        if iter_lines:
            loss = float(iter_lines[-1])

        # Processes alive
        r = subprocess.run(["ps", "aux"], capture_output=True, text=True)
        alive = sum(1 for l in r.stdout.splitlines() if "flower_pCE" in l)

        # History
        if not history or history[-1][0] != current_round:
            history.append((current_round, dice, loss))

        # Display
        clear_output(wait=True)
        pct   = current_round / FL_ROUNDS * 100
        bar   = "█" * int(pct / 2) + "░" * (50 - int(pct / 2))
        eta_s = int((FL_ROUNDS - current_round) * 10)
        eta   = f"{eta_s//3600}h {(eta_s%3600)//60}m {eta_s%60}s"

        print("=" * 62)
        print("       FedLPPA Prostate MRI — Live Training")
        print("=" * 62)
        print(f"  Progress : |{bar}| {pct:.1f}%")
        print(f"  Round    : {current_round} / {FL_ROUNDS}")
        print(f"  Iter     : {current_round * 10} / {MAX_ITER}")
        print(f"  Loss     : {loss:.4f}")
        print(f"  Dice     : {dice:.4f}{'  ← non-zero soon!' if dice == 0 and current_round >= 30 else ''}")
        print(f"  Procs    : {alive} alive")
        print(f"  ETA      : {eta}")
        print("=" * 62)

        if len(history) > 1:
            print(f"\n  {'Round':<8} {'Loss':<12} {'Dice':<8} {'Bar'}")
            print(f"  {'─'*40}")
            for rnd, d, l in history[-12:]:
                bar_d = "█" * int(d * 30) if d > 0 else "·"
                flag  = " ✓" if d > 0 else ""
                print(f"  {rnd:<8} {l:<12.4f} {d:<8.4f} {bar_d}{flag}")

        # Error checks
        warns = []
        for cn in ["client1","client2","client4","client5","client6"]:
            with open(f"{LOG_DIR}/{cn}.log") as f:
                clog = f.read()
            if "OutOfMemoryError" in clog: warns.append(f"OOM:{cn}")
            if "DataLoader worker" in clog: warns.append(f"DL crash:{cn}")
        if warns:
            print(f"\n  ⚠  {' | '.join(warns)}")

        if current_round >= FL_ROUNDS:
            print("\n  ✓ Training complete!")
            break
        if alive == 0:
            print("\n  ✗ Sab processes mar gaye — check logs")
            break

    except KeyboardInterrupt:
        clear_output(wait=True)
        print("Cell roka — training background mein chal rahi hai.")
        print("Check karo:  !ps aux | grep flower_pCE")
        break
    except Exception as e:
        print(f"Monitor error: {e}")
        time.sleep(5)

       FedLPPA Prostate MRI — Live Training
  Progress : |██████████████████████████████████████████████████| 100.0%
  Round    : 300 / 300
  Iter     : 3000 / 3000
  Loss     : 0.0475
  Dice     : 0.3370
  Procs    : 12 alive
  ETA      : 0h 0m 0s

  Round    Loss         Dice     Bar
  ────────────────────────────────────────
  284      0.0993       0.3486   ██████████ ✓
  286      0.0864       0.3293   █████████ ✓
  288      0.0567       0.3503   ██████████ ✓
  289      0.0873       0.2802   ████████ ✓
  290      0.0617       0.2802   ████████ ✓
  292      0.0694       0.2410   ███████ ✓
  294      0.0629       0.1493   ████ ✓
  295      0.0799       0.2615   ███████ ✓
  296      0.0635       0.2615   ███████ ✓
  298      0.1104       0.2950   ████████ ✓
  299      0.1104       0.3370   ██████████ ✓
  300      0.0475       0.3370   ██████████ ✓

  ✓ Training complete!


In [96]:
import os
LOG_DIR = "/content/FedLPPA/logs/prostate"
with open(f"{LOG_DIR}/client1.log") as f:
    lines = [l for l in f.read().splitlines() if "iteration" in l]
for l in lines[-10:]:
    print(l)

INFO flwr 2026-05-29 20:10:47,337 | flower_pCE_2D_v4_FedLPPA.py:319 | client 0 : iteration 271 : lr: 0.002756, loss : 0.336560, loss_ce: 0.210783
         False, False, False, False]], device='cuda:0')INFO flwr 2026-05-29 20:10:47,695 | flower_pCE_2D_v4_FedLPPA.py:319 | client 0 : iteration 272 : lr: 0.002755, loss : 0.526481, loss_ce: 0.326461
INFO flwr 2026-05-29 20:10:48,055 | flower_pCE_2D_v4_FedLPPA.py:319 | client 0 : iteration 273 : lr: 0.002754, loss : 0.485679, loss_ce: 0.356683
         False, False, False, False]], device='cuda:0')INFO flwr 2026-05-29 20:10:48,414 | flower_pCE_2D_v4_FedLPPA.py:319 | client 0 : iteration 274 : lr: 0.002753, loss : 0.326783, loss_ce: 0.163151
INFO flwr 2026-05-29 20:10:48,682 | flower_pCE_2D_v4_FedLPPA.py:319 | client 0 : iteration 275 : lr: 0.002752, loss : 0.340313, loss_ce: 0.168458
         False, False, False, False]], device='cuda:0')INFO flwr 2026-05-29 20:10:50,692 | flower_pCE_2D_v4_FedLPPA.py:319 | client 0 : iteration 276 : lr: 0.00

In [92]:
import os
LOG_DIR = "/content/FedLPPA/logs/prostate"

# Server progress
with open(f"{LOG_DIR}/server.log") as f:
    content = f.read()
tqdm_lines = [l for l in content.splitlines() if "/300 [" in l]
print("Server progress:", tqdm_lines[-1].strip() if tqdm_lines else "none")

# All clients last line
for cname in ["client1","client2","client4","client5","client6"]:
    with open(f"{LOG_DIR}/{cname}.log") as f:
        lines = [l for l in f.read().splitlines()
                 if l.strip() and "False" not in l]
    print(f"{cname}: {lines[-1][:100] if lines else 'empty'}")

Server progress: 2%|▌                                | 5/300 [01:00<51:11, 10.41s/it]DEBUG flwr 2026-05-29 19:50:18,850 | server.py:218 | fit_round 60: strategy sampled 5 clients (out of 5)
client1:   with autocast(enabled=self.amp):
client2:   with autocast(enabled=self.amp):
client4:   with autocast(enabled=self.amp):
client5:   with autocast(enabled=self.amp):
client6:   with autocast(enabled=self.amp):


In [67]:
import subprocess, os

# Check if server/clients still running
r = subprocess.run(["ps", "aux"], capture_output=True, text=True)
flower = [l for l in r.stdout.splitlines() if "flower_pCE" in l]
print(f"Live flower processes: {len(flower)}")
for l in flower:
    print(" ", l[:120])

# Check round progress
LOG_DIR = "/content/FedLPPA/logs/prostate"
with open(f"{LOG_DIR}/server.log") as f:
    content = f.read()

rounds_done = content.count("client_0_lr")
metric_lines = [l for l in content.splitlines() if "mean_dice" in l]
print(f"\nRounds done : {rounds_done} / 1000")
print(f"Iters done  : {rounds_done * 10} / 10000")
if metric_lines:
    print(f"Latest      : {metric_lines[-1].strip()}")

Live flower processes: 0

Rounds done : 5 / 1000
Iters done  : 50 / 10000
Latest      : INFO flwr 2026-05-29 18:53:38,314 | flower_common_v4.py:519 | iteration 50 : mean_dice : 0.0; mean_hd95 : 0.0; mean_recall : 0.0; mean_precision : 0.0; mean_jc : 0.0; mean_specificity : 0.0; mean_ravd : 0.0;


In [68]:
# ── Patch flower_pCE_2D_v4_FedLPPA.py to handle AMP NaN safely ────
fpath = "/content/FedLPPA/code_v4/flower_pCE_2D_v4_FedLPPA.py"
with open(fpath) as f:
    src = f.read()

# Replace bare scaler.step/update with NaN-safe version
OLD = """                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()"""

NEW = """                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                # Skip step if gradients contain NaN/Inf
                found_inf = False
                for p in model.parameters():
                    if p.grad is not None:
                        if not torch.isfinite(p.grad).all():
                            found_inf = True
                            break
                if not found_inf:
                    scaler.step(optimizer)
                scaler.update()"""

if OLD in src:
    src = src.replace(OLD, NEW)
    with open(fpath, "w") as f:
        f.write(src)
    print("GradScaler NaN patch applied.")
else:
    print("Pattern not found — checking actual scaler usage...")
    # Find all scaler.step lines
    for i, l in enumerate(src.splitlines()):
        if "scaler" in l:
            print(f"  line {i}: {l}")

GradScaler NaN patch applied.


In [62]:
# ── Correct live monitor ───────────────────────────────────────────
import os

LOG_DIR = "/content/FedLPPA/logs/prostate"

with open(f"{LOG_DIR}/server.log") as f:
    content = f.read()

# Each completed round prints client_0_lr once
rounds_done = content.count("client_0_lr")
print(f"Rounds completed : {rounds_done} / 1000")
print(f"Iterations done  : {rounds_done * 10} / 10000")

# Latest dice from server
metric_lines = [l for l in content.splitlines() if "mean_dice" in l]
if metric_lines:
    print(f"\nLatest: {metric_lines[-1].strip()}")
else:
    print("Metrics: 0.0 (normal — model still in cold start, check after ~300 iters)")

# Client alive check
print("\nClient status:")
for cname in ["client1","client2","client4","client5","client6"]:
    with open(f"{LOG_DIR}/{cname}.log") as f:
        clog = f.read()
    lines = [l for l in clog.splitlines() if l.strip() and "DEBUG" not in l]
    last  = lines[-1][:100] if lines else "empty"
    print(f"  {cname}: {last}")

Rounds completed : 5 / 1000
Iterations done  : 50 / 10000

Latest: INFO flwr 2026-05-29 18:53:38,314 | flower_common_v4.py:519 | iteration 50 : mean_dice : 0.0; mean_hd95 : 0.0; mean_recall : 0.0; mean_precision : 0.0; mean_jc : 0.0; mean_specificity : 0.0; mean_ravd : 0.0;

Client status:
  client1: KeyboardInterrupt
  client2: KeyboardInterrupt
  client4: KeyboardInterrupt
  client5: KeyboardInterrupt
  client6: RuntimeError: DataLoader worker (pid(s) 60614) exited unexpectedly


In [63]:
import os, subprocess

LOG_DIR = "/content/FedLPPA/logs/prostate"

with open(f"{LOG_DIR}/server.log") as f:
    content = f.read()

rounds_done = content.count("client_0_lr")
print(f"Rounds : {rounds_done} / 1000  |  Iters : {rounds_done * 10} / 10000")

metric_lines = [l for l in content.splitlines() if "mean_dice" in l]
if metric_lines:
    last = metric_lines[-1]
    # Parse out just the numbers
    import re
    nums = re.findall(r'(\w+) : ([\d.]+)', last)
    print("\nLatest metrics:")
    for k, v in nums:
        bar = "█" * int(float(v) * 20) if float(v) > 0 else "░" * 20
        print(f"  {k:<20} {float(v):.4f}  {bar}")
else:
    print("Metrics: all zero — normal before iter ~300")

# Check which processes are alive
r = subprocess.run(["ps", "aux"], capture_output=True, text=True)
flower = [l for l in r.stdout.splitlines() if "flower_pCE" in l]
print(f"\nLive flower processes: {len(flower)}")

print("\nClient logs (last line):")
for cname in ["client1","client2","client4","client5","client6"]:
    with open(f"{LOG_DIR}/{cname}.log") as f:
        lines = [l for l in f.read().splitlines()
                 if l.strip() and "DEBUG" not in l]
    status = "DEAD ✗" if "KeyboardInterrupt" in "\n".join(lines[-3:]) else "alive ✓"
    last   = lines[-1][:80] if lines else "empty"
    print(f"  {cname} [{status}]: {last}")

Rounds : 5 / 1000  |  Iters : 50 / 10000

Latest metrics:
  mean_dice            0.0000  ░░░░░░░░░░░░░░░░░░░░
  mean_hd95            0.0000  ░░░░░░░░░░░░░░░░░░░░
  mean_recall          0.0000  ░░░░░░░░░░░░░░░░░░░░
  mean_precision       0.0000  ░░░░░░░░░░░░░░░░░░░░
  mean_jc              0.0000  ░░░░░░░░░░░░░░░░░░░░
  mean_specificity     0.0000  ░░░░░░░░░░░░░░░░░░░░
  mean_ravd            0.0000  ░░░░░░░░░░░░░░░░░░░░

Live flower processes: 0

Client logs (last line):
  client1 [DEAD ✗]: KeyboardInterrupt
  client2 [DEAD ✗]: KeyboardInterrupt
  client4 [DEAD ✗]: KeyboardInterrupt
  client5 [DEAD ✗]: KeyboardInterrupt
  client6 [alive ✓]: RuntimeError: DataLoader worker (pid(s) 60614) exited unexpectedly


In [64]:
# ── Patch val_2D.py for prostate single-channel input ─────────────
fpath = "/content/FedLPPA/code_v4/val_2D.py"
with open(fpath) as f:
    src = f.read()

# test_single_volume expects (C,H,W) input but prostate slices are (H,W)
OLD = "def test_single_volume(case, net, classes, patch_size=[256, 256]"
NEW = "def test_single_volume(case, net, classes, patch_size=[256, 256]"

# Check if img_class is passed through
print("val_2D.py test_single_volume signature:")
for i, l in enumerate(src.splitlines()):
    if "def test_single_volume" in l or "def test_single_volume_ds" in l:
        print(f"  line {i}: {l}")

print("\nFirst 60 lines of test_single_volume:")
inside = False
count  = 0
for l in src.splitlines():
    if "def test_single_volume" in l:
        inside = True
    if inside:
        print(l)
        count += 1
        if count > 60:
            break

val_2D.py test_single_volume signature:
  line 28: def test_single_volume(image, label, net, classes, patch_size=[256, 256], amp=False):
  line 80: def test_single_volume_nomain(image, label, net, classes, patch_size=[256, 256], amp=False):
  line 131: def test_single_volume_ds(image, label, net, classes, patch_size=[256, 256]):
  line 169: def test_single_volume_cct(image, label, net, classes, patch_size=[256, 256]):
  line 260: def test_single_volume_tel(image, label, net, classes, patch_size=[256, 256]):

First 60 lines of test_single_volume:
def test_single_volume(image, label, net, classes, patch_size=[256, 256], amp=False):
    image, label = image.squeeze(1).squeeze(0).cpu().detach(
    ).numpy(), label.squeeze(1).squeeze(0).cpu().detach().numpy()
    # ###odoc val
    if len(image.shape) == 3:
        prediction = np.zeros_like(label)
        
        slice = image
            # x, y = slice.shape[0], slice.shape[1]
            # slice = zoom(
            #     slice, (patch_si

In [65]:
# ── Full diagnosis ────────────────────────────────────────────────
with open("/content/FedLPPA/logs/prostate/server.log") as f:
    content = f.read()

lines = content.splitlines()
print(f"Total log lines : {len(lines)}")
print(f"Last 40 lines:\n")
print("\n".join(lines[-40:]))

Total log lines : 171
Last 40 lines:

client_1_lr
client_2_lr
client_3_lr
client_4_lr
client_0_lr
client_1_lr
client_2_lr
client_3_lr
client_4_lr
Traceback (most recent call last):
  File "/content/FedLPPA/code_v4/flower_pCE_2D_v4_FedLPPA.py", line 742, in <module>
    main()
  File "/content/FedLPPA/code_v4/flower_pCE_2D_v4_FedLPPA.py", line 730, in main
    fl.server.start_server(
  File "/usr/local/lib/python3.12/dist-packages/flwr/server/app.py", line 176, in start_server
    hist = _fl(
           ^^^^
  File "/usr/local/lib/python3.12/dist-packages/flwr/server/app.py", line 217, in _fl
    hist = server.fit(num_rounds=config.num_rounds, timeout=config.round_timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/FedLPPA/code_v4/flower_common_v4.py", line 402, in fit
    res_fit = self.fit_round(server_round=current_round, timeout=timeout)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/loca

In [66]:
with open("/content/FedLPPA/logs/prostate/server.log") as f:
    print(f.read())

INFO flwr 2026-05-29 18:52:21,021 | flower_pCE_2D_v4_FedLPPA.py:628 | Arguments: Namespace(server_address='127.0.0.1:8095', gpu=0, role='server', iters=10, eval_iters=10, tsne_iters=200, rep_iters=12, sample_fraction=1.0, min_num_clients=5, strategy='FedUniV2.1', mu=0.001, lam=0.5, sort_lam=0.5, sort_beta=0.5, beta=0.5, alpha=0.1, init_iters=100, common_iters=400, pretrain_iters=150, pretrain=0, img_size=384, model_momentum=0.5, prompt='universal', attention='dual', dual_init='aggregated', label_prompt=1, cid=0, root_path='../data/Prostate_h5', exp='prostate/FedLPPA', client='client_all', sup_type='mask', sup_type_list=None, model='unet_univ5', num_classes=2, max_iterations=10000, batch_size=6, in_chns=1, deterministic=1, amp=0, base_lr=0.001, patch_size=[256, 256], img_class='prostate', seed=2022, snapshot_path='../model/prostate/FedLPPA')
INFO flwr 2026-05-29 18:52:30,453 | app.py:148 | Starting Flower server, config: ServerConfig(num_rounds=10000, round_timeout=None)
INFO flwr 2026-

In [40]:
# ── Create empty Domain3 placeholder (no I2CVB data) ─────────────
import os

H5_BASE = "/content/data/Prostate_h5"

for split in ["train", "test"]:
    path = os.path.join(H5_BASE, "Domain3", split)
    os.makedirs(path, exist_ok=True)
    print(f"Created: {path}")

# Verify all 6 domains now exist
for dom in ["Domain1","Domain2","Domain3","Domain4","Domain5","Domain6"]:
    for split in ["train","test"]:
        path = os.path.join(H5_BASE, dom, split)
        n = len(os.listdir(path)) if os.path.exists(path) else "MISSING"
        print(f"  {dom}/{split}: {n} files")

Created: /content/data/Prostate_h5/Domain3/train
Created: /content/data/Prostate_h5/Domain3/test
  Domain1/train: 334 files
  Domain1/test: 87 files
  Domain2/train: 282 files
  Domain2/test: 87 files
  Domain3/train: 0 files
  Domain3/test: 0 files
  Domain4/train: 188 files
  Domain4/test: 64 files
  Domain5/train: 116 files
  Domain5/test: 42 files
  Domain6/train: 299 files
  Domain6/test: 69 files


In [37]:
# ── Patch flower_pCE_2D_v4_FedLPPA.py for prostate support ───────
fpath = "/content/FedLPPA/code_v4/flower_pCE_2D_v4_FedLPPA.py"

with open(fpath) as f:
    src = f.read()

# Fix 1: expand the assert to include prostate
src = src.replace(
    "assert args.img_class in ['odoc', 'faz', 'polyp']",
    "assert args.img_class in ['odoc', 'faz', 'polyp', 'prostate']"
)

# Fix 2: wherever img_class branch checks miss prostate, add it
# The volume_batch unsqueeze path (faz branch) is correct for prostate
src = src.replace(
    "if self.args.img_class == 'faz':\n                volume_batch, label_batch = sampled_batch['image'].unsqueeze(1), sampled_batch['label']",
    "if self.args.img_class in ['faz', 'prostate']:\n                volume_batch, label_batch = sampled_batch['image'].unsqueeze(1), sampled_batch['label']"
)
src = src.replace(
    "if args.img_class == 'faz':\n        volume_batch, label_batch = sampled_batch['image'].unsqueeze(1), sampled_batch['label']",
    "if args.img_class in ['faz', 'prostate']:\n        volume_batch, label_batch = sampled_batch['image'].unsqueeze(1), sampled_batch['label']"
)

with open(fpath, "w") as f:
    f.write(src)
print("Patched assert and img_class branches.")

# ── Verify the assert line is fixed ──────────────────────────────
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-c",
     "import sys; sys.path.insert(0, '.');"
     "import argparse;"
     "sys.argv = ['x', '--img_class', 'prostate', '--role', 'server',"
     "            '--root_path', '.', '--exp', 'test',"
     "            '--server_address', '127.0.0.1:9999'];"
     "exec(open('flower_pCE_2D_v4_FedLPPA.py').read().split('def main')[0]);"
     "print('assert patch OK')"],
    capture_output=True, text=True,
    cwd="/content/FedLPPA/code_v4"
)
# Just grep for the patched line directly
with open(fpath) as f:
    lines = [l.strip() for l in f if "assert args.img_class" in l]
print("Assert line now reads:", lines)
print("\nRe-run training cell.")

Patched assert and img_class branches.
Assert line now reads: ["assert args.img_class in ['odoc', 'faz', 'polyp', 'prostate']"]

Re-run training cell.


In [27]:
# ── Install last missing package ──────────────────────────────────
!pip install -q efficientnet-pytorch

# ── Patch tree_filter imports in both common files ────────────────
# The wheel is py3.9-only; wrap the import in try/except so the
# rest of the file loads cleanly. FedLPPA (FedUniV2.1 strategy)
# does not use TreeEnergy at runtime — only the older FedICRA path did.

import re, os

PATCH = '''\
try:
    from utils.TreeEnergyLoss.kernels.lib_tree_filter.modules.tree_filter import MinimumSpanningTree
    from utils.TreeEnergyLoss.kernels.lib_tree_filter.modules.tree_filter import TreeFilter2D
    TREE_FILTER_AVAILABLE = True
except Exception:
    MinimumSpanningTree = None
    TreeFilter2D = None
    TREE_FILTER_AVAILABLE = False
'''

FILES_TO_PATCH = [
    "/content/FedLPPA/code_v4/flower_common_v4.py",
    "/content/FedLPPA/code_v4/flower_common_v4_addprostate.py",
]

BAD_LINE_1 = ("from utils.TreeEnergyLoss.kernels.lib_tree_filter"
              ".modules.tree_filter import MinimumSpanningTree\n")
BAD_LINE_2 = ("from utils.TreeEnergyLoss.kernels.lib_tree_filter"
              ".modules.tree_filter import TreeFilter2D\n")

for fpath in FILES_TO_PATCH:
    with open(fpath, "r") as f:
        src = f.read()

    if "TREE_FILTER_AVAILABLE" in src:
        print(f"  already patched: {os.path.basename(fpath)}")
        continue

    # Remove both bad lines and insert the try/except block once
    src = src.replace(BAD_LINE_1, "")
    src = src.replace(BAD_LINE_2, PATCH)

    with open(fpath, "w") as f:
        f.write(src)
    print(f"  patched: {os.path.basename(fpath)}")

# ── Final import check before launching training ──────────────────
print("\nVerifying all imports resolve...")
import subprocess, sys, os

check = subprocess.run(
    [sys.executable, "-c",
     "import sys; sys.path.insert(0,'.');"
     "from networks.net_factory import net_factory;"
     "from tensorboardX import SummaryWriter;"
     "from info_nce import InfoNCE;"
     "from medpy import metric;"
     "print('ALL OK')"],
    capture_output=True, text=True,
    cwd="/content/FedLPPA/code_v4"
)
print(check.stdout)
if check.returncode != 0:
    print("ERRORS:")
    print(check.stderr[-2000:])

  Preparing metadata (setup.py) ... done
  patched: flower_common_v4.py
  patched: flower_common_v4_addprostate.py

Verifying all imports resolve...
ALL OK



## Monitoring — run this cell anytime during training

In [30]:
# ════════════════════════════════════════════════════════════════
# FIX 1 — flwr imports tensorflow at startup via tensorboard util.
# Stub that file out so flwr never touches tensorflow.
# ════════════════════════════════════════════════════════════════
import os

tb_stub = "/usr/local/lib/python3.12/dist-packages/flwr/server/utils/tensorboard.py"
with open(tb_stub, "w") as f:
    f.write("# stubbed — tensorflow removed from flwr startup path\n")
    f.write("def tensorboard(*args, **kwargs):\n    pass\n")
print("flwr tensorboard stub written")

# ════════════════════════════════════════════════════════════════
# FIX 2 — flower_common.py (NOT flower_common_v4.py) also has the
# hard tree_filter import and was not patched earlier.
# Patch all three files that import from tree_filter directly.
# ════════════════════════════════════════════════════════════════
PATCH = '''\
try:
    from utils.TreeEnergyLoss.kernels.lib_tree_filter.modules.tree_filter import MinimumSpanningTree
    from utils.TreeEnergyLoss.kernels.lib_tree_filter.modules.tree_filter import TreeFilter2D
    TREE_FILTER_AVAILABLE = True
except Exception:
    MinimumSpanningTree = None
    TreeFilter2D = None
    TREE_FILTER_AVAILABLE = False
'''

BAD_1 = ("from utils.TreeEnergyLoss.kernels.lib_tree_filter"
         ".modules.tree_filter import MinimumSpanningTree\n")
BAD_2 = ("from utils.TreeEnergyLoss.kernels.lib_tree_filter"
         ".modules.tree_filter import TreeFilter2D\n")

FILES = [
    "/content/FedLPPA/code_v4/flower_common.py",          # missed earlier
    "/content/FedLPPA/code_v4/flower_common_v4.py",
    "/content/FedLPPA/code_v4/flower_common_v4_addprostate.py",
]

for fpath in FILES:
    if not os.path.exists(fpath):
        print(f"  not found: {fpath}")
        continue
    with open(fpath) as f:
        src = f.read()
    if "TREE_FILTER_AVAILABLE" in src:
        print(f"  already patched: {os.path.basename(fpath)}")
        continue
    src = src.replace(BAD_1, "")
    src = src.replace(BAD_2, PATCH)
    with open(fpath, "w") as f:
        f.write(src)
    print(f"  patched: {os.path.basename(fpath)}")

# ════════════════════════════════════════════════════════════════
# Quick smoke-test — if this prints OK, re-run training cell
# ════════════════════════════════════════════════════════════════
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-c",
     "import sys; sys.path.insert(0,'.');"
     "import flwr;"
     "from flower_common import PretrainDataset;"
     "from flower_common_v4 import BaseClient;"
     "print('ALL OK')"],
    capture_output=True, text=True,
    cwd="/content/FedLPPA/code_v4"
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-2000:])

flwr tensorboard stub written
  patched: flower_common.py
  already patched: flower_common_v4.py
  already patched: flower_common_v4_addprostate.py
ALL OK



In [22]:
# ── Install all missing FedLPPA dependencies ─────────────────────
!pip install -q \
    tensorboardX \
    info-nce-pytorch \
    medpy \
    scikit-learn \
    seaborn \
    tqdm \
    matplotlib

# Verify all critical imports resolve
import importlib, sys

checks = [
    "tensorboardX",
    "info_nce",
    "medpy",
    "sklearn",
    "seaborn",
    "tqdm",
    "flwr",
    "torch",
    "torchvision",
    "numpy",
    "scipy",
    "cv2",
    "h5py",
    "SimpleITK",
]

all_ok = True
for pkg in checks:
    try:
        importlib.import_module(pkg)
        print(f"  OK  {pkg}")
    except ImportError as e:
        print(f"  MISSING  {pkg}  -> {e}")
        all_ok = False

print("\nAll OK — proceed to training." if all_ok else
      "\nSome packages still missing — check above.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  OK  tensorboardX
  OK  info_nce
  OK  medpy
  OK  sklearn
  OK  seaborn
  OK  tqdm
  OK  flwr
  OK  torch
  OK  torchvision
  OK  numpy
  OK  scipy
  OK  cv2
  OK  h5py
  OK  SimpleITK

All OK — proceed to training.


In [20]:
# ── Read full server log ──────────────────────────────────────────
with open("/content/FedLPPA/logs/prostate/server.log") as f:
    print(f.read())

Traceback (most recent call last):
  File "/content/FedLPPA/code_v4/flower_pCE_2D_v4_FedLPPA.py", line 19, in <module>
    from info_nce import InfoNCE
ModuleNotFoundError: No module named 'info_nce'



In [21]:
# ── Read full server log ──────────────────────────────────────────
with open("/content/FedLPPA/logs/prostate/server.log") as f:
    print(f.read())

Traceback (most recent call last):
  File "/content/FedLPPA/code_v4/flower_pCE_2D_v4_FedLPPA.py", line 19, in <module>
    from info_nce import InfoNCE
ModuleNotFoundError: No module named 'info_nce'



In [33]:
# ── Step 1: Verify both fixes are in place ────────────────────────
import os

# Check tensorboard stub
tb = "/usr/local/lib/python3.12/dist-packages/flwr/server/utils/tensorboard.py"
with open(tb) as f:
    content = f.read()
print("tensorboard stub OK:", "tensorflow" not in content)

# Check flower_common.py patch
fc = "/content/FedLPPA/code_v4/flower_common.py"
with open(fc) as f:
    content = f.read()
print("flower_common.py patched:", "TREE_FILTER_AVAILABLE" in content)

# If either is False, re-apply fixes
if "tensorflow" in open(tb).read():
    with open(tb, "w") as f:
        f.write("# stubbed\ndef tensorboard(*args, **kwargs):\n    pass\n")
    print("  -> tensorboard stub re-applied")

PATCH = '''\
try:
    from utils.TreeEnergyLoss.kernels.lib_tree_filter.modules.tree_filter import MinimumSpanningTree
    from utils.TreeEnergyLoss.kernels.lib_tree_filter.modules.tree_filter import TreeFilter2D
    TREE_FILTER_AVAILABLE = True
except Exception:
    MinimumSpanningTree = None
    TreeFilter2D = None
    TREE_FILTER_AVAILABLE = False
'''
BAD_1 = "from utils.TreeEnergyLoss.kernels.lib_tree_filter.modules.tree_filter import MinimumSpanningTree\n"
BAD_2 = "from utils.TreeEnergyLoss.kernels.lib_tree_filter.modules.tree_filter import TreeFilter2D\n"

for fpath in [
    "/content/FedLPPA/code_v4/flower_common.py",
    "/content/FedLPPA/code_v4/flower_common_v4.py",
    "/content/FedLPPA/code_v4/flower_common_v4_addprostate.py",
]:
    with open(fpath) as f: src = f.read()
    if "TREE_FILTER_AVAILABLE" not in src:
        src = src.replace(BAD_1, "").replace(BAD_2, PATCH)
        with open(fpath, "w") as f: f.write(src)
        print(f"  -> patched: {os.path.basename(fpath)}")

# ── Step 2: Clear stale logs ──────────────────────────────────────
import glob
for lf in glob.glob("/content/FedLPPA/logs/prostate/*.log"):
    open(lf, "w").close()
print("Logs cleared.")

# ── Step 3: Final smoke test ──────────────────────────────────────
import subprocess, sys
r = subprocess.run(
    [sys.executable, "-c",
     "import sys; sys.path.insert(0,'.');"
     "import flwr; print('flwr OK');"
     "from flower_common import PretrainDataset; print('flower_common OK');"
     "from flower_common_v4 import BaseClient; print('flower_common_v4 OK');"
     "from networks.net_factory import net_factory; print('net_factory OK');"
     "print('ALL CLEAR - run training cell now')"],
    capture_output=True, text=True,
    cwd="/content/FedLPPA/code_v4"
)
print(r.stdout)
if r.returncode != 0:
    print("STILL FAILING:")
    print(r.stderr[-3000:])

tensorboard stub OK: False
flower_common.py patched: True
  -> tensorboard stub re-applied
Logs cleared.
flwr OK
flower_common OK
flower_common_v4 OK
net_factory OK
ALL CLEAR - run training cell now

